# Live ASL Fingerspelling Recognition with a PyTorch CNN

**Image and Video Processing project: static American Sign Language letters, from dataset to webcam**

This notebook trains a convolutional neural network (CNN) on the *Sign Language MNIST* dataset and then uses it to recognise fingerspelled letters live from a laptop webcam. Training runs on the GPU with PyTorch. Every image processing step (cropping, grayscale conversion, contrast enhancement, resizing, intensity scaling, geometric and photometric augmentation, blurring, Grad-CAM upsampling) is written out in the code and explained in the markdown, rather than hidden inside a library call.

### The live pipeline

```
  webcam frame (BGR, e.g. 640 x 480)
        |
        v
  hand detection (MediaPipe)  ---> 21 hand landmarks in pixel coordinates
        |
        v
  crop: square box around the landmarks + padding, clamped to the frame
        |
        v
  preprocessing: grayscale -> optional CLAHE -> resize to 28 x 28 (INTER_AREA) -> scale to [0, 1]
        |
        v
  CNN on the GPU  ---> 24 class probabilities (softmax)
        |
        v
  temporal smoothing: average of the last 10 frames, shown only if confidence >= 0.70
        |
        v
  smoothed letter on screen (+ top-3 alternatives, FPS, word builder)
```

### An important limitation

The model classifies **single still images**. The letters **J** and **Z** are signed with a movement (tracing a "J" or a "Z" in the air), so they cannot be recognised from one frame and are not in the dataset. The model therefore knows 24 letters: A to Y without J. Recognising J and Z would need a sequence model (for example a model over landmark trajectories), which is listed under next steps.

### Contents

1. Title and overview
2. Setup: imports, versions, seeds and `CONFIG`
3. GPU check
4. Load and inspect the data
5. Image preprocessing
6. Train/validation split
7. GPU data augmentation
8. Baseline model
9. CNN architecture
10. Training
11. Diagnosing the fit
12. Final test evaluation (run once)
13. Robustness check
14. Grad-CAM
15. Saving and loading the model
16. Live webcam demo
17. Optional fine-tuning on your own images
18. Conclusion

**How to run.** Set `DATA_DIR` and `CAM_INDEX` in section 2, then *Run All*. Training takes a few minutes on a laptop GPU. Section 16 opens an OpenCV window and keeps running until you press **q** in that window. Section 17 only captures images and fine-tunes when you switch it on in `CONFIG`.

## 2. Setup

**What.** Import every library, print its version, fix the random seeds and collect every hyperparameter, threshold and switch in a single `CONFIG` dictionary.

**Why.** Printing versions makes the run reproducible and makes bug reports meaningful: MediaPipe and PyTorch in particular change their APIs between releases. Keeping all settings in one dictionary means there are no "magic numbers" scattered through the code, the whole experiment can be saved to JSON alongside the weights, and changing an experiment means changing one place.

In [ ]:
# --- Standard library -------------------------------------------------------
import contextlib
import copy
import json
import math
import os
import platform
import random
import sys
import time
import urllib.request
from collections import deque
from pathlib import Path

# --- Scientific stack -------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

# --- Vision and deep learning -----------------------------------------------
import cv2
import mediapipe as mp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

print(f"Python        : {sys.version.split()[0]}  ({platform.system()} {platform.release()})")
for name, module in [
    ("torch", torch),
    ("torchvision", torchvision),
    ("opencv-python", cv2),
    ("mediapipe", mp),
    ("numpy", np),
    ("pandas", pd),
    ("matplotlib", matplotlib),
    ("seaborn", sns),
    ("scikit-learn", sklearn),
]:
    print(f"{name:<14}: {getattr(module, '__version__', 'unknown')}")

# Which MediaPipe hand-tracking API does this installation provide?
try:
    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision as mp_vision
    HAS_MP_TASKS = True
except ImportError:
    HAS_MP_TASKS = False
HAS_MP_SOLUTIONS = hasattr(mp, "solutions") and hasattr(getattr(mp, "solutions"), "hands")
print(f"\nMediaPipe Tasks HandLandmarker available : {HAS_MP_TASKS}")
print(f"MediaPipe legacy mp.solutions.hands      : {HAS_MP_SOLUTIONS}")

The next cell holds the only two values you should need to edit, `DATA_DIR` and `CAM_INDEX`, followed by `CONFIG`. Each entry has a comment saying what it controls.

In [ ]:
# ============================================================================
# The only two settings you should need to edit
# ============================================================================
DATA_DIR = Path("dataset")   # folder containing sign_mnist_train.csv and sign_mnist_test.csv
CAM_INDEX = 0                # 0 is normally the built-in laptop webcam; try 1, 2 ... otherwise

# ============================================================================
# Every hyperparameter, threshold and toggle in one place
# ============================================================================
CONFIG = {
    # --- Reproducibility and hardware --------------------------------------
    "SEED": 42,
    "REQUIRE_GPU": True,            # stop with an explanation if no GPU is found
    "USE_AMP": True,                # float16 mixed precision (only ever used on CUDA)
    "CUDNN_BENCHMARK": True,        # faster convolutions, slightly less reproducible
    "SMOKE_TEST_SIZE": 1024,        # matrix size for the GPU smoke test

    # --- Dataset facts (checked with asserts) -------------------------------
    "TRAIN_CSV": "sign_mnist_train.csv",
    "TEST_CSV": "sign_mnist_test.csv",
    "IMG_SIZE": 28,                 # images are 28 x 28 grayscale
    "PIXEL_MAX": 255.0,             # uint8 maximum, used to scale to [0, 1]
    "NUM_CLASSES": 24,              # A-Y without J (Z is also absent)
    "MISSING_LABEL": 9,             # J never appears
    "MAX_RAW_LABEL": 24,            # raw labels run 0..24
    "EXPECTED_TRAIN_ROWS": 27455,
    "EXPECTED_TEST_ROWS": 7172,
    "ROW_COUNT_TOLERANCE": 0.01,    # accept +-1 % in case of a slightly different download

    # --- Split ----------------------------------------------------------------
    "VAL_FRACTION": 0.15,           # stratified 85/15 split of the training CSV

    # --- GPU augmentation ---------------------------------------------------
    "AUGMENT": True,
    "AUG_ROTATION_DEG": 10.0,       # rotation sampled from +-10 degrees
    "AUG_TRANSLATE_FRAC": 0.10,     # translation sampled from +-10 % of the image width/height
    "AUG_SCALE_RANGE": (0.9, 1.1),  # zoom factor
    "AUG_HFLIP": False,             # horizontal flip changes handedness, so off by default
    "AUG_HFLIP_PROB": 0.5,          # used only if AUG_HFLIP is True
    "AUG_BRIGHTNESS": 0.10,         # additive brightness shift sampled from +-0.10
    "AUG_CONTRAST_RANGE": (0.8, 1.2),  # contrast factor around each image's mean
    "AUG_NOISE_STD": 0.02,          # standard deviation of additive Gaussian noise
    "AUG_PADDING_MODE": "border",   # how grid_sample fills pixels that come from outside the image
    "N_AUG_EXAMPLES": 15,           # augmented copies shown in the demo grid

    # --- Baseline -------------------------------------------------------------
    "BASELINE_EPOCHS": 15,
    "BASELINE_LR": 1e-3,

    # --- CNN architecture ---------------------------------------------------
    "CONV_FILTERS": (32, 64, 128),
    "CONV_KERNEL": 3,
    "CONV_DROPOUT": 0.25,
    "DENSE_UNITS": 256,
    "DENSE_DROPOUT": 0.45,

    # --- Training -------------------------------------------------------------
    "BATCH_SIZE": 256,
    "EVAL_BATCH_SIZE": 2048,
    "MAX_EPOCHS": 40,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "LABEL_SMOOTHING": 0.05,
    "LR_FACTOR": 0.5,               # ReduceLROnPlateau: multiply the LR by this ...
    "LR_PATIENCE": 3,               # ... after this many epochs without val-loss improvement
    "MIN_LR": 1e-6,
    "EARLY_STOP_PATIENCE": 7,
    "EARLY_STOP_MIN_DELTA": 1e-4,   # smallest val-loss drop that counts as an improvement
    "TRACK_CLEAN_TRAIN_ACC": True,  # also measure accuracy on the un-augmented training split

    # --- Fit diagnosis and targets -------------------------------------------
    "UNDERFIT_TRAIN_ACC": 0.95,
    "OVERFIT_GAP": 0.03,            # train - val accuracy
    "VAL_RISE_MIN_EPOCHS": 4,       # "rose steadily" needs at least this many epochs after the minimum ...
    "VAL_RISE_FRACTION": 0.75,      # ... with at least this fraction of epoch-to-epoch increases ...
    "VAL_RISE_REL": 0.10,           # ... and a total rise of at least 10 % above the minimum
    "TREND_WINDOW": 3,              # epochs used to judge whether val loss is flat/falling
    "TARGET_TEST_ACC": 0.98,
    "TARGET_MACRO_F1": 0.98,

    # --- Evaluation displays --------------------------------------------------
    "TOP_CONFUSED_PAIRS": 10,
    "N_MISCLASSIFIED_SHOW": 24,
    "GRID_COLS": 8,

    # --- Robustness check -----------------------------------------------------
    "ROBUST_BRIGHTNESS": (-0.3, -0.15, 0.15, 0.3),
    "ROBUST_BLUR_SIGMA": (0.5, 1.0, 1.5),
    "ROBUST_ROTATION_DEG": (-15.0, -7.5, 7.5, 15.0),
    "ROBUST_NOISE_STD": (0.05, 0.10, 0.20),
    "BLUR_RADIUS_SIGMAS": 3,        # Gaussian kernel half-width in standard deviations

    # --- Grad-CAM -------------------------------------------------------------
    "GRADCAM_N_CORRECT": 4,
    "GRADCAM_N_WRONG": 4,
    "GRADCAM_ALPHA": 0.5,           # heatmap opacity in the overlay

    # --- Saving -----------------------------------------------------------------
    "OUTPUT_DIR": "artifacts",
    "MODEL_FILE": "sign_cnn_best.pt",
    "FINETUNED_MODEL_FILE": "sign_cnn_finetuned.pt",
    "LABEL_MAP_FILE": "label_map.json",
    "CONFIG_FILE": "config.json",

    # --- Hand detection (MediaPipe) --------------------------------------------
    "MAX_HANDS": 1,
    "HAND_DETECTION_CONF": 0.6,
    "HAND_TRACKING_CONF": 0.5,
    "HAND_MODEL_PATH": "hand_landmarker.task",
    "HAND_MODEL_URL": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
                      "hand_landmarker/float16/latest/hand_landmarker.task",

    # --- Live demo ------------------------------------------------------------------
    "RUN_LIVE_DEMO": True,          # set False to skip the webcam cell when running all
    "CAM_WIDTH": 640,
    "CAM_HEIGHT": 480,
    "BOX_PADDING": 0.25,            # crop side = hand size * (1 + BOX_PADDING)
    "USE_CLAHE": False,             # starting state; toggle live with the h key
    "CLAHE_CLIP_LIMIT": 2.0,
    "CLAHE_TILE_GRID": 4,
    "MIRROR_INPUT": False,          # starting state; toggle live with the m key
    "INSET_SIZE": 140,              # enlarged 28 x 28 model input shown in the corner
    "INSET_MARGIN": 10,
    "SMOOTHING_WINDOW": 10,         # number of probability vectors averaged
    "CONF_THRESHOLD": 0.70,         # below this the display shows "?"
    "TOP_K": 3,
    "HOLD_SECONDS": 1.0,            # hold a letter this long to type it
    "PANEL_HEIGHT": 190,            # black information panel under the video
    "FPS_SMOOTHING": 0.9,           # exponential moving average factor for the FPS counter
    "WINDOW_NAME": "ASL fingerspelling - press q to quit",

    # --- Capture and fine-tuning ------------------------------------------------
    "RUN_CAPTURE": False,           # set True to record your own images (section 17)
    "RUN_FINETUNE": True,           # fine-tune only if enough of your images exist
    "MY_DATA_DIR": "my_data",
    "CAPTURE_LETTERS": "ABCDEFGHIKLMNOPQRSTUVWXY",
    "CAPTURE_PER_LETTER": 150,
    "CAPTURE_EVERY_N_FRAMES": 2,    # skip frames so consecutive saves differ a little
    "FT_MIN_IMAGES": 200,           # do not fine-tune on fewer of your images than this
    "FT_HOLDOUT_FRAC": 0.20,        # of your images, kept aside and only used for the final report
    "FT_VAL_FRAC": 0.15,            # of the remainder, used for early stopping
    "FT_MY_REPEAT": 5,              # oversample your images this many times in the mix
    "FT_ORIG_PER_MY": 2,            # original images per (repeated) image of yours in the mix
    "FT_LR": 1e-4,
    "FT_EPOCHS": 15,
    "FT_PATIENCE": 4,
}

def seed_everything(seed: int) -> None:
    """Seed Python, NumPy and PyTorch (CPU and every CUDA device)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)   # safe to call even without CUDA

seed_everything(CONFIG["SEED"])
torch.backends.cudnn.benchmark = CONFIG["CUDNN_BENCHMARK"]

# Letters are used as class names throughout; built here so it is available everywhere.
print(f"DATA_DIR = {DATA_DIR.resolve()}")
print(f"CAM_INDEX = {CAM_INDEX}")
print(f"{len(CONFIG)} settings in CONFIG, seed = {CONFIG['SEED']}")

**Note on `cudnn.benchmark`.** With `benchmark = True`, cuDNN times several convolution algorithms on the first batches and keeps the fastest one for each input shape. That speeds training up, but the chosen algorithm can differ between runs and some algorithms are not bit-for-bit deterministic. Seeding Python, NumPy and PyTorch makes the *data* (split, shuffling, augmentation) repeatable; the final accuracy may still vary by a few tenths of a percent from run to run. For exact reproducibility you would set `benchmark = False` and `torch.use_deterministic_algorithms(True)`, and accept slower training.

## 3. GPU check

**What.** Choose the compute device in the order CUDA (NVIDIA), then MPS (Apple Silicon), then CPU. Print the details of the PyTorch build and the GPU, then run a small matrix multiplication on the chosen device as a smoke test.

**Why.** The most common reason a "GPU notebook" silently runs on the CPU is that `pip install torch` installed a **CPU-only build** of PyTorch (this is the default on Windows). In that case `torch.version.cuda` is `None` and `torch.cuda.is_available()` is `False`, even though the laptop has an NVIDIA GPU and a working driver. With `CONFIG["REQUIRE_GPU"] = True` the notebook stops here with an explanation instead of training slowly on the CPU without telling you.

In [ ]:
def select_device() -> torch.device:
    """Return the best available device: CUDA, then Apple MPS, then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def sync_device(device: torch.device) -> None:
    """Wait for queued GPU work to finish, so that timings are honest."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


DEVICE = select_device()

print(f"torch.__version__         : {torch.__version__}")
print(f"torch.version.cuda        : {torch.version.cuda}")
print(f"torch.cuda.is_available() : {torch.cuda.is_available()}")
print(f"Selected device           : {DEVICE}")

if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(DEVICE)
    print(f"GPU name                  : {props.name}")
    print(f"Total VRAM                : {props.total_memory / 1024**3:.2f} GiB")
    print(f"Compute capability        : {props.major}.{props.minor}")
    print(f"cuDNN version             : {torch.backends.cudnn.version()}")
elif DEVICE.type == "mps":
    print("Apple Silicon GPU (Metal Performance Shaders) in use.")

if DEVICE.type == "cpu":
    if torch.version.cuda is None:
        cause = (
            "This PyTorch build has no CUDA support (torch.version.cuda is None). The most likely cause\n"
            "is that a CPU-only build of PyTorch was installed, which is what a plain `pip install torch`\n"
            "gives you on Windows. Uninstall it and install the CUDA build using the exact command shown at\n"
            "https://pytorch.org/get-started/locally/ (choose your OS, Pip, Python and a CUDA version), e.g.\n"
            "    pip uninstall -y torch torchvision\n"
            "    pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128\n"
            "then restart the Jupyter kernel."
        )
    else:
        cause = (
            f"PyTorch was built with CUDA {torch.version.cuda}, but no GPU is visible. Check that the NVIDIA\n"
            "driver is installed and recent enough (run `nvidia-smi` in a terminal), that the laptop is not\n"
            "forcing integrated graphics, and that CUDA_VISIBLE_DEVICES is not set to an empty value."
        )
    if CONFIG["REQUIRE_GPU"]:
        raise RuntimeError("No GPU found and CONFIG['REQUIRE_GPU'] is True.\n\n" + cause +
                           "\n\nTo run on the CPU anyway (slowly), set CONFIG['REQUIRE_GPU'] = False.")
    print("WARNING: no GPU found, continuing on the CPU because REQUIRE_GPU is False.\n" + cause)

# Mixed precision is only used on CUDA (float16 autocast on MPS/CPU is not worth it here).
USE_AMP = bool(CONFIG["USE_AMP"] and DEVICE.type == "cuda")
print(f"Mixed precision (AMP)     : {USE_AMP}")
print(f"cudnn.benchmark           : {torch.backends.cudnn.benchmark}")


def autocast_context():
    """float16 autocast on CUDA when AMP is on, otherwise a do-nothing context."""
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()

In [ ]:
# Smoke test: one matrix multiplication on the chosen device, checked against the CPU result.
n = CONFIG["SMOKE_TEST_SIZE"]
a = torch.randn(n, n, device=DEVICE)
b = torch.randn(n, n, device=DEVICE)
_ = a @ b                      # warm-up (first call initialises the GPU libraries)
sync_device(DEVICE)
t0 = time.perf_counter()
c = a @ b
sync_device(DEVICE)
elapsed_ms = (time.perf_counter() - t0) * 1000

max_diff = (c.cpu() - a.cpu() @ b.cpu()).abs().max().item()
print(f"{n}x{n} matmul on {c.device}: {elapsed_ms:.2f} ms, max |GPU - CPU| = {max_diff:.2e}")
del a, b, c

**Reading the output.** On CUDA you should see `torch.version.cuda` set to a version number, the GPU name and its VRAM, and a matmul that takes well under a millisecond to a few milliseconds. The small difference against the CPU result (around 1e-4 or less) comes from floating-point operations being summed in a different order on the GPU; it is expected and harmless.

## 4. Load and inspect the data

**What.** Read the two CSV files, check the facts we rely on with `assert` statements, build the mapping between class indices and letters, and look at the data: class balance, one example per letter and the distribution of pixel intensities.

**Why.** Every later step assumes these facts (784 pixel columns, 24 classes, label 9 missing). If a file is truncated or a different version of the dataset is downloaded, it is better to fail loudly here than to get a quietly wrong model.

**The labels.** The raw labels are the alphabet positions 0 (A) to 24 (Y). Label 9 (J) never appears, and Z (25) is absent too, because both letters involve motion. A network needs contiguous class indices `0..23`, so labels are remapped: 0-8 stay the same and 10-24 become 9-23. The dictionary `IDX_TO_LETTER` turns a class index back into a letter.

In [ ]:
train_csv = DATA_DIR / CONFIG["TRAIN_CSV"]
test_csv = DATA_DIR / CONFIG["TEST_CSV"]
for path in (train_csv, test_csv):
    assert path.exists(), f"Missing {path}. Download Sign Language MNIST from Kaggle and set DATA_DIR."

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
print(f"train: {train_df.shape}   test: {test_df.shape}")

S = CONFIG["IMG_SIZE"]
expected_cols = ["label"] + [f"pixel{i}" for i in range(1, S * S + 1)]
for name, df, expected_rows in [("train", train_df, CONFIG["EXPECTED_TRAIN_ROWS"]),
                                 ("test", test_df, CONFIG["EXPECTED_TEST_ROWS"])]:
    assert list(df.columns) == expected_cols, f"{name}: columns are not label, pixel1..pixel{S*S}"
    assert df.isna().sum().sum() == 0, f"{name}: contains missing values"
    assert abs(len(df) - expected_rows) <= CONFIG["ROW_COUNT_TOLERANCE"] * expected_rows, \
        f"{name}: {len(df)} rows, expected about {expected_rows}"
    pixels = df.iloc[:, 1:].to_numpy()
    assert pixels.min() >= 0 and pixels.max() <= CONFIG["PIXEL_MAX"], f"{name}: pixels outside 0..255"
    labels = set(df["label"].unique())
    assert CONFIG["MISSING_LABEL"] not in labels, f"{name}: label {CONFIG['MISSING_LABEL']} (J) should not appear"
    assert labels <= set(range(CONFIG["MAX_RAW_LABEL"] + 1)), f"{name}: labels outside 0..{CONFIG['MAX_RAW_LABEL']}"
    print(f"{name}: {len(df)} rows, {len(labels)} classes, missing values: {df.isna().sum().sum()}")

RAW_LABELS = sorted(train_df["label"].unique())
assert len(RAW_LABELS) == CONFIG["NUM_CLASSES"], "expected 24 classes in the training set"
assert set(test_df["label"].unique()) == set(RAW_LABELS), "train and test must contain the same classes"

# Remap raw labels (0..24 without 9) to contiguous indices 0..23, and keep a way back to the letter.
RAW_TO_IDX = {raw: idx for idx, raw in enumerate(RAW_LABELS)}
IDX_TO_LETTER = {idx: chr(ord("A") + raw) for idx, raw in enumerate(RAW_LABELS)}
LETTER_TO_IDX = {letter: idx for idx, letter in IDX_TO_LETTER.items()}
LETTERS = [IDX_TO_LETTER[i] for i in range(CONFIG["NUM_CLASSES"])]
assert "J" not in LETTERS and "Z" not in LETTERS
print("Class index -> letter:", IDX_TO_LETTER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=False)
for ax, (name, df) in zip(axes, [("Training set", train_df), ("Test set", test_df)]):
    counts = df["label"].map(RAW_TO_IDX).value_counts().sort_index()
    ax.bar(LETTERS, counts.values, color="steelblue")
    ax.axhline(counts.mean(), color="black", linestyle="--", linewidth=1, label=f"mean = {counts.mean():.0f}")
    ax.set_title(f"{name}: images per letter (n = {len(df):,})")
    ax.set_xlabel("Letter")
    ax.set_ylabel("Number of images")
    ax.legend()
plt.tight_layout()
plt.show()

train_counts = train_df["label"].value_counts()
print(f"Training images per class: min {train_counts.min()}, max {train_counts.max()}, "
      f"ratio max/min = {train_counts.max() / train_counts.min():.2f}")

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(10, 7.5))
for ax, idx in zip(axes.flat, range(CONFIG["NUM_CLASSES"])):
    raw = RAW_LABELS[idx]
    row = train_df[train_df["label"] == raw].iloc[0, 1:].to_numpy(dtype=np.uint8).reshape(S, S)
    ax.imshow(row, cmap="gray", vmin=0, vmax=CONFIG["PIXEL_MAX"])
    ax.set_title(f"{IDX_TO_LETTER[idx]} (raw label {raw})", fontsize=9)
    ax.set_xlabel("x (pixels)", fontsize=7)
    ax.set_ylabel("y (pixels)", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("One training example per letter (28 x 28 grayscale)")
plt.tight_layout()
plt.show()

In [ ]:
train_pixels = train_df.iloc[:, 1:].to_numpy(dtype=np.uint8).ravel()
test_pixels = test_df.iloc[:, 1:].to_numpy(dtype=np.uint8).ravel()
bins = np.arange(int(CONFIG["PIXEL_MAX"]) + 2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_pixels, bins=bins, density=True, alpha=0.6, label="train")
ax.hist(test_pixels, bins=bins, density=True, alpha=0.6, label="test")
ax.set_title("Pixel intensity distribution")
ax.set_xlabel("Pixel intensity (0 = black, 255 = white)")
ax.set_ylabel("Density")
ax.legend()
plt.show()
print(f"train mean {train_pixels.mean():.1f}, std {train_pixels.std():.1f} | "
      f"test mean {test_pixels.mean():.1f}, std {test_pixels.std():.1f}")
del train_pixels, test_pixels

**Interpreting the output.**

- All asserts pass: 27,455 training and 7,172 test rows, 784 pixels each, no missing values, 24 classes with J absent.
- The classes are roughly balanced (the largest class has under twice as many images as the smallest), so plain accuracy is a reasonable headline metric. We still report macro F1, which weights every letter equally.
- The sample grid shows a close-cropped hand, roughly centred, on a fairly plain background, with the camera facing the signer. These are the properties our webcam crops must imitate.
- Intensities cover the whole 0-255 range with most mass in the mid-greys. Train and test have very similar histograms, so there is no large brightness shift between the two files. A webcam can easily produce one, which is why section 13 tests brightness changes.

## 5. Image preprocessing

**What.** A single function converts a grayscale `uint8` image into exactly what the network expects: a `float32` tensor of shape `(1, 1, 28, 28)` (batch, channel, height, width) with values in `[0, 1]`, on the chosen device. The scaling itself lives in one helper, `scale_pixels`, which is used both to build the training tensors and inside `preprocess_for_model` for the webcam.

**Why preprocessing at inference must match training exactly.** The network learns weights for one particular input distribution. If it was trained on values in `[0, 1]` and is then given values in `[0, 255]`, every activation is 255 times too large, the batch-normalisation statistics stored during training no longer apply, and the predictions are meaningless, even though nothing raises an error. Smaller mismatches matter too: a different resize interpolation, an inverted image, a colour image instead of grayscale, or a crop that is mirrored relative to the training photos all shift the input away from what the model has seen. Sharing one function for training and inference removes a whole class of these bugs.

**The scaling.** For each pixel value $p \in \{0, \dots, 255\}$:

$$x = \frac{p}{255}, \qquad x \in [0, 1].$$

We deliberately do not subtract a dataset mean or divide by a standard deviation. That would be a statistic *fitted* on data (and would have to be computed on the training split only, to avoid leakage). Here it is unnecessary because every convolution is followed by batch normalisation, which learns its own normalisation from the training batches.

In [ ]:
def scale_pixels(images_uint8: np.ndarray, device: torch.device = DEVICE) -> torch.Tensor:
    """Convert uint8 images (any leading shape, last two dims H x W) to float32 in [0, 1] on `device`.

    This is the single source of truth for intensity scaling: the training tensors and the
    webcam input both go through it. The uint8 array is copied to the device first (4x less
    data than float32) and converted there.
    """
    assert images_uint8.dtype == np.uint8, f"expected uint8, got {images_uint8.dtype}"
    tensor = torch.from_numpy(np.ascontiguousarray(images_uint8)).to(device)
    return tensor.to(torch.float32).div_(CONFIG["PIXEL_MAX"])


def preprocess_for_model(gray_img: np.ndarray, device: torch.device = DEVICE) -> torch.Tensor:
    """Turn one grayscale uint8 image into the model input: float32 (1, 1, 28, 28) in [0, 1].

    If the image is not already 28 x 28 it is shrunk with INTER_AREA (area averaging), which is
    the right interpolation for downsampling because it averages all source pixels that fall
    into each output pixel instead of sampling a few of them, so it does not alias.
    """
    assert gray_img.ndim == 2, f"expected a single-channel image, got shape {gray_img.shape}"
    size = CONFIG["IMG_SIZE"]
    if gray_img.shape != (size, size):
        gray_img = cv2.resize(gray_img, (size, size), interpolation=cv2.INTER_AREA)
    return scale_pixels(gray_img, device).reshape(1, 1, size, size)


def dataframe_to_arrays(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Split a Sign MNIST dataframe into uint8 images (N, 28, 28) and remapped labels (N,)."""
    size = CONFIG["IMG_SIZE"]
    images = df.iloc[:, 1:].to_numpy(dtype=np.uint8).reshape(-1, size, size)
    labels = df["label"].map(RAW_TO_IDX).to_numpy(dtype=np.int64, copy=True)   # writable copy
    return images, labels

In [ ]:
all_train_images, all_train_labels = dataframe_to_arrays(train_df)
test_images, test_labels = dataframe_to_arrays(test_df)

example = all_train_images[0]
x_single = preprocess_for_model(example)
x_batch = scale_pixels(all_train_images[:8]).unsqueeze(1)    # the path used for training tensors

print(f"input : dtype {example.dtype}, shape {example.shape}, range [{example.min()}, {example.max()}]")
print(f"output: dtype {x_single.dtype}, shape {tuple(x_single.shape)}, device {x_single.device}, "
      f"range [{x_single.min().item():.3f}, {x_single.max().item():.3f}]")
assert torch.equal(x_single[0], x_batch[0]), "webcam path and training path must give identical tensors"
print("The single-image (webcam) path and the batch (training) path give identical tensors.")

# The same function also handles a larger crop, as the webcam will produce.
big = cv2.resize(example, (4 * S, 4 * S), interpolation=cv2.INTER_CUBIC)
print(f"a {big.shape} crop becomes {tuple(preprocess_for_model(big).shape)}")

**Interpreting the output.** The output tensor has the right type, shape, device and range, and the `assert` confirms that the single-image path (used live) and the batch path (used for training) give bit-identical results. A larger crop is shrunk to 28 x 28 automatically, so the webcam code cannot accidentally feed the network the wrong size.

## 6. Train/validation split

**What.** Split the training CSV 85/15 with `train_test_split(..., stratify=labels)`, so each letter keeps the same proportion in both parts. Then convert each part to `float32` tensors once and keep them on the GPU for the whole run.

**Why keep the data on the GPU.** The whole dataset is about 34,600 images of 28 x 28 floats, roughly 110 MB, which fits easily in laptop VRAM. With a normal `DataLoader`, every batch is assembled on the CPU and copied over PCIe to the GPU, and for tiny images that copy and the Python overhead cost more than the forward pass. Here a batch is simply `X_train[idx]`, where `idx` comes from `torch.randperm` *on the GPU*, so there is no copy at all. This also avoids `DataLoader` worker processes, which are a common source of hangs and errors inside Jupyter on Windows (workers are started with `spawn` and cannot see functions defined in the notebook).

**Caveat about validation.** The Kaggle training set was made by augmenting a much smaller set of original photos, so many training images are near-duplicates of each other. A random split puts near-copies of validation images into the training set, so **validation accuracy will be too optimistic**. We use validation only for early stopping and the learning-rate schedule. The official test set, built from different originals, is the honest hold-out. It is used once, in section 12, and never for tuning.

In [ ]:
train_idx, val_idx = train_test_split(
    np.arange(len(all_train_labels)),
    test_size=CONFIG["VAL_FRACTION"],
    stratify=all_train_labels,
    random_state=CONFIG["SEED"],
)

# Convert once, keep on the device. Shapes become (N, 1, 28, 28): one grayscale channel.
X_train = scale_pixels(all_train_images[train_idx]).unsqueeze(1)
y_train = torch.from_numpy(all_train_labels[train_idx]).to(DEVICE)
X_val = scale_pixels(all_train_images[val_idx]).unsqueeze(1)
y_val = torch.from_numpy(all_train_labels[val_idx]).to(DEVICE)
X_test = scale_pixels(test_images).unsqueeze(1)
y_test = torch.from_numpy(test_labels).to(DEVICE)

total_mb = sum(t.element_size() * t.nelement() for t in (X_train, y_train, X_val, y_val, X_test, y_test)) / 2**20
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:<5}: X {tuple(X.shape)} {X.dtype} on {X.device} | y {tuple(y.shape)} {y.dtype}")
print(f"All tensors together: {total_mb:.1f} MB on {DEVICE}")

# Stratification check: the class proportions in train and val should match closely.
p_train = torch.bincount(y_train, minlength=CONFIG["NUM_CLASSES"]).float() / len(y_train)
p_val = torch.bincount(y_val, minlength=CONFIG["NUM_CLASSES"]).float() / len(y_val)
print(f"Largest difference in class proportion between train and val: {(p_train - p_val).abs().max().item():.4f}")


def iterate_minibatches(n: int, batch_size: int, shuffle: bool, device: torch.device = DEVICE):
    """Yield index tensors for mini-batches. Shuffling uses torch.randperm on the device itself."""
    order = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]

**Interpreting the output.** All six tensors are on the GPU, take roughly 100 MB together, and the class proportions in train and validation agree to within a fraction of a percent, which is what stratification guarantees. From here on, batching is just indexing into these tensors.

## 7. GPU data augmentation

**What.** During training, every batch is randomly transformed on the GPU: a small rotation, translation and zoom (geometric), then a brightness and contrast change and a little Gaussian noise (photometric). Each image in the batch gets **its own** random parameters.

**Why.** Augmentation shows the network plausible variations of each training image, so it has to learn the hand shape rather than memorise exact pixel positions. This is also our first step towards webcam images, which will never be centred and lit exactly like the dataset. We do not use `torchvision.transforms` on a whole batch, because those transforms draw one set of random parameters and apply it to every image in the batch, which gives much less variety.

### The affine matrix maths

A 2-D affine transformation maps a point $\mathbf{p} = (x, y)^\top$ to $\mathbf{p}' = A\mathbf{p} + \mathbf{t}$, where $A$ is a $2 \times 2$ matrix (rotation, scale, shear, flip) and $\mathbf{t}$ a translation. Written as one $2 \times 3$ matrix acting on homogeneous coordinates $(x, y, 1)^\top$:

$$
\theta = \begin{bmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \end{bmatrix},
\qquad
\begin{bmatrix} x' \\ y' \end{bmatrix} = \theta \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}.
$$

For a rotation by angle $\varphi$ combined with a uniform scale factor $s$:

$$
A = \frac{1}{s}\begin{bmatrix} \cos\varphi & -\sin\varphi \\ \sin\varphi & \cos\varphi \end{bmatrix}.
$$

Two details of PyTorch's `F.affine_grid` explain the $1/s$:

1. **Normalised coordinates.** Pixel positions are expressed in $[-1, 1]$, with $(-1, -1)$ at the top-left corner and $(1, 1)$ at the bottom-right. The image is 2 units wide, so a translation of 10 % of the width is $t_x = 0.2$. The same matrix therefore works for any image size.
2. **Backward (inverse) mapping.** $\theta$ maps each pixel of the **output** image to the position in the **input** image it should be read from. This is how image warping is normally done, because every output pixel then gets exactly one value, with no holes and no pixels written twice. As a consequence, $\theta$ is the *inverse* of the visual transformation: to make the hand look $s$ times larger, each output pixel must read from a point $1/s$ times as far from the centre, hence the $1/s$. In the same way a positive $t_x$ moves the content left, and the visible rotation is $-\varphi$. Because we draw $\varphi$, $t$ and $s$ from ranges symmetric about "no change", the inverse transformation has (almost exactly) the same distribution, so we can sample $\theta$ directly.

A horizontal flip is $x \mapsto -x$, i.e. multiplying the first column of $A$ by $-1$. It is off by default (`AUG_HFLIP`) because flipping turns a right-handed sign into a left-handed one, and some letters are distinguished by orientation.

`F.affine_grid` turns the batch of $\theta$ matrices, shape $(N, 2, 3)$, into a sampling grid of shape $(N, 28, 28, 2)$. `F.grid_sample` then reads the input at those (generally non-integer) positions with **bilinear interpolation**: for a sampling point at fractional offsets $(\alpha, \beta)$ between four neighbouring pixels $I_{00}, I_{10}, I_{01}, I_{11}$,

$$
I(\alpha, \beta) = (1-\alpha)(1-\beta)\,I_{00} + \alpha(1-\beta)\,I_{10} + (1-\alpha)\beta\,I_{01} + \alpha\beta\,I_{11}.
$$

Points that fall outside the image take the value of the nearest border pixel (`padding_mode="border"`), which matches the plain backgrounds better than filling with black.

### Photometric jitter

For image $i$ with mean intensity $\mu_i$, contrast factor $c_i$, brightness shift $b_i$ and per-pixel noise $n \sim \mathcal{N}(0, \sigma^2)$:

$$
x' = \operatorname{clamp}\big(c_i\,(x - \mu_i) + \mu_i + b_i + n,\; 0,\; 1\big).
$$

Scaling around the mean changes contrast without changing the average brightness; $b_i$ then changes the brightness; the clamp keeps values in the valid range the network was trained on.

In [ ]:
def build_affine_matrices(angles_rad: torch.Tensor, scales: torch.Tensor,
                          translations: torch.Tensor, flips: torch.Tensor | None = None) -> torch.Tensor:
    """Build one 2 x 3 sampling matrix theta per image for F.affine_grid.

    angles_rad: (N,) rotation angles; scales: (N,) zoom factors; translations: (N, 2) in normalised
    units (2 = full image width); flips: optional (N,) bool, True = mirror horizontally.
    theta = [[cos/s, -sin/s, tx], [sin/s, cos/s, ty]]  (see the markdown above for why it is 1/s).
    """
    cos, sin = torch.cos(angles_rad), torch.sin(angles_rad)
    inv_s = 1.0 / scales
    theta = torch.zeros(angles_rad.shape[0], 2, 3, device=angles_rad.device, dtype=angles_rad.dtype)
    theta[:, 0, 0] = cos * inv_s
    theta[:, 0, 1] = -sin * inv_s
    theta[:, 1, 0] = sin * inv_s
    theta[:, 1, 1] = cos * inv_s
    theta[:, :, 2] = translations
    if flips is not None:
        theta[:, :, 0] = torch.where(flips[:, None], -theta[:, :, 0], theta[:, :, 0])  # x -> -x
    return theta


def apply_affine(x: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """Warp a batch (N, C, H, W) with per-image matrices theta (N, 2, 3) using bilinear sampling."""
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode=CONFIG["AUG_PADDING_MODE"], align_corners=False)


def uniform(n: int, low: float, high: float, device: torch.device) -> torch.Tensor:
    """n samples from U(low, high) on `device`."""
    return low + (high - low) * torch.rand(n, device=device)


def gpu_augment(x: torch.Tensor, cfg: dict = CONFIG) -> torch.Tensor:
    """Augment a batch (N, 1, H, W) in [0, 1] on its own device, with different random parameters per image."""
    n, device = x.shape[0], x.device

    # --- Geometric: rotation, zoom, translation (and optional flip) in one affine warp ----------
    max_angle = np.deg2rad(cfg["AUG_ROTATION_DEG"])
    angles = uniform(n, -max_angle, max_angle, device)
    scales = uniform(n, *cfg["AUG_SCALE_RANGE"], device)
    max_shift = 2.0 * cfg["AUG_TRANSLATE_FRAC"]             # normalised units: the image is 2 wide
    translations = torch.stack([uniform(n, -max_shift, max_shift, device),
                                uniform(n, -max_shift, max_shift, device)], dim=1)
    flips = (torch.rand(n, device=device) < cfg["AUG_HFLIP_PROB"]) if cfg["AUG_HFLIP"] else None
    x = apply_affine(x, build_affine_matrices(angles, scales, translations, flips))

    # --- Photometric: contrast around each image's mean, brightness shift, Gaussian noise ---------
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    contrast = uniform(n, *cfg["AUG_CONTRAST_RANGE"], device).view(n, 1, 1, 1)
    brightness = uniform(n, -cfg["AUG_BRIGHTNESS"], cfg["AUG_BRIGHTNESS"], device).view(n, 1, 1, 1)
    x = contrast * (x - mean) + mean + brightness
    x = x + cfg["AUG_NOISE_STD"] * torch.randn_like(x)
    return x.clamp_(0.0, 1.0)

In [ ]:
# Show one image augmented many times *in a single batch*: every copy should look different,
# which proves the parameters are drawn per image, not per batch.
k = CONFIG["N_AUG_EXAMPLES"]
source = X_train[:1]
augmented = gpu_augment(source.repeat(k, 1, 1, 1))
assert augmented.device == source.device and augmented.shape == (k, 1, S, S)

cols = 4
rows = int(np.ceil((k + 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(8, 2.2 * rows))
images_to_show = torch.cat([source, augmented]).squeeze(1).cpu().numpy()   # CPU only for plotting
for i, ax in enumerate(axes.flat):
    ax.axis("off")
    if i < len(images_to_show):
        ax.imshow(images_to_show[i], cmap="gray", vmin=0, vmax=1)
        ax.set_title("original" if i == 0 else f"augmented {i}", fontsize=9)
fig.suptitle(f"GPU augmentation of one '{IDX_TO_LETTER[y_train[0].item()]}' (per-image random parameters)")
plt.tight_layout()
plt.show()

# Timing: how expensive is augmentation for a full training batch?
batch = X_train[:CONFIG["BATCH_SIZE"]]
_ = gpu_augment(batch); sync_device(DEVICE)
t0 = time.perf_counter()
for _ in range(10):
    _ = gpu_augment(batch)
sync_device(DEVICE)
print(f"Augmenting a batch of {len(batch)} on {DEVICE}: {(time.perf_counter() - t0) / 10 * 1000:.2f} ms")

**Interpreting the output.** Every augmented copy has a different rotation, position, zoom, brightness and contrast, so the parameters really are per image. The hand stays recognisable, which is the point: augmentation should produce variations a human would still label the same way. If the copies looked unrecognisable, the ranges in `CONFIG` would be too aggressive. Augmenting a whole batch on the GPU takes well under a millisecond, so it adds almost nothing to training time.

## 8. Baseline: multinomial logistic regression

**What.** Before building a CNN, train the simplest sensible model: logistic regression, i.e. a single linear layer from the 784 pixels to 24 logits, followed by softmax. It is trained in PyTorch on the GPU without augmentation.

**Why.** A baseline tells us how hard the problem is and how much the CNN actually adds. Logistic regression treats each pixel independently and has no notion of neighbouring pixels or of shapes that move, so we expect it to do well on the near-duplicate validation set and noticeably worse on the honest test set. Its test accuracy is reported alongside the CNN's in section 12 (the test set is still only looked at once, at the end).

In [ ]:
class LogisticRegression(nn.Module):
    """Softmax regression: flatten the image and apply one linear layer (softmax is in the loss)."""

    def __init__(self, n_inputs: int, n_classes: int):
        super().__init__()
        self.linear = nn.Linear(n_inputs, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x.flatten(start_dim=1))


@torch.no_grad()
def predict_logits(model: nn.Module, X: torch.Tensor, batch_size: int = CONFIG["EVAL_BATCH_SIZE"]) -> torch.Tensor:
    """Return float32 logits for every row of X (kept on the device). Puts the model in eval mode."""
    model.eval()
    outputs = []
    for idx in iterate_minibatches(len(X), batch_size, shuffle=False, device=X.device):
        with autocast_context():
            outputs.append(model(X[idx]).float())
    return torch.cat(outputs)


seed_everything(CONFIG["SEED"])
baseline = LogisticRegression(S * S, CONFIG["NUM_CLASSES"]).to(DEVICE)
baseline_opt = torch.optim.AdamW(baseline.parameters(), lr=CONFIG["BASELINE_LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
ce_loss = nn.CrossEntropyLoss()

t0 = time.perf_counter()
for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    baseline.train()
    for idx in iterate_minibatches(len(X_train), CONFIG["BATCH_SIZE"], shuffle=True):
        loss = ce_loss(baseline(X_train[idx]), y_train[idx])
        baseline_opt.zero_grad(set_to_none=True)
        loss.backward()
        baseline_opt.step()
sync_device(DEVICE)

baseline_train_acc = (predict_logits(baseline, X_train).argmax(1) == y_train).float().mean().item()
baseline_val_acc = (predict_logits(baseline, X_val).argmax(1) == y_val).float().mean().item()
print(f"Logistic regression after {CONFIG['BASELINE_EPOCHS']} epochs ({time.perf_counter() - t0:.1f} s)")
print(f"  train accuracy: {baseline_train_acc:.2%}")
print(f"  val accuracy  : {baseline_val_acc:.2%}")

**Interpreting the output.** In the reference run, logistic regression reached about 80 % on both training and validation data. Training and validation agree because the validation images are near-copies of training images, so this number says little about new images. A linear model has one fixed template per letter and cannot cope with the hand being shifted or rotated by a few pixels. Section 12 shows how much of this survives on the test set (about 63 % in the reference run), which is the real comparison with the CNN.

## 9. CNN architecture

**What.** A compact VGG-style CNN with three convolutional blocks (32, 64 and 128 filters) and a small classifier on top.

```
input 1 x 28 x 28
 -> block 1: [Conv3x3 -> BatchNorm -> ReLU] x 2, MaxPool 2x2, Dropout   -> 32 x 14 x 14
 -> block 2: [Conv3x3 -> BatchNorm -> ReLU] x 2, MaxPool 2x2, Dropout   -> 64 x 7 x 7
 -> block 3: [Conv3x3 -> BatchNorm -> ReLU] x 2, MaxPool 2x2, Dropout   -> 128 x 4 x 4
 -> flatten (2048) -> Linear 256 -> ReLU -> Dropout 0.45 -> Linear 24 (logits)
```

**What each layer type does.**

- **`Conv2d` (3 x 3, padding 1).** Slides a set of learned 3 x 3 filters over the image. Each filter computes a weighted sum of a pixel's neighbourhood, so it responds to a local pattern such as an edge, a corner or a finger outline, wherever it appears (weight sharing makes the network largely tolerant to translation). Two stacked 3 x 3 convolutions see a 5 x 5 region with fewer parameters than one 5 x 5 convolution and an extra non-linearity. Padding of 1 keeps the height and width unchanged. The bias is switched off because the following batch-norm layer has its own shift, which would make a bias redundant.
- **`BatchNorm2d`.** Normalises each channel to zero mean and unit variance over the batch, then applies a learned scale and shift. This keeps activations in a stable range, allows a higher learning rate and acts as a mild regulariser. In `eval()` mode it uses running averages collected during training instead of batch statistics, which is why switching modes correctly matters.
- **`ReLU`.** $\max(0, z)$: the non-linearity that lets stacked layers represent more than a single linear function.
- **`MaxPool2d(2)`.** Keeps the largest value in each 2 x 2 window, halving the resolution. This makes the features more tolerant of small shifts and lets later layers see larger parts of the hand (their *receptive field* grows). We use `ceil_mode=True`: 7 is odd, and the default (floor) would turn the 7 x 7 map into 3 x 3 by silently **discarding the last row and column**, i.e. the bottom and right edges of the image would never reach the classifier. Rounding up (7 -> 4) keeps every pixel in play. An earlier version of this notebook used the default, and the Grad-CAM maps in section 14 exposed the problem as bright artefacts along those edges.
- **`Dropout`.** During training, randomly sets a fraction of activations to zero (and rescales the rest), so the network cannot rely on any single feature. It is switched off in `eval()` mode.
- **`Linear` layers.** The flattened 128 x 4 x 4 feature map is combined into 256 hidden units and then 24 outputs, one *logit* per letter. Softmax turns logits into probabilities; it is applied inside `CrossEntropyLoss` during training and explicitly at inference time.

In [ ]:
class ConvBlock(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2, then MaxPool 2x2 and Dropout."""

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, dropout: float):
        super().__init__()
        padding = kernel_size // 2   # "same" padding for odd kernels
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2, ceil_mode=True),   # ceil: 7 -> 4, so the last row/column is not dropped
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SignCNN(nn.Module):
    """Three conv blocks and a dense classifier producing one logit per letter."""

    def __init__(self, num_classes: int, filters: tuple = (32, 64, 128), kernel_size: int = 3,
                 conv_dropout: float = 0.25, dense_units: int = 256, dense_dropout: float = 0.45,
                 img_size: int = 28, in_channels: int = 1):
        super().__init__()
        blocks, channels = [], in_channels
        for out_channels in filters:
            blocks.append(ConvBlock(channels, out_channels, kernel_size, conv_dropout))
            channels = out_channels
        self.features = nn.Sequential(*blocks)
        spatial = img_size
        for _ in filters:                                # each block halves H and W, rounding up
            spatial = math.ceil(spatial / 2)             # 28 -> 14 -> 7 -> 4
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels * spatial * spatial, dense_units),
            nn.ReLU(),
            nn.Dropout(dense_dropout),
            nn.Linear(dense_units, num_classes),
        )

    @property
    def last_conv(self) -> nn.Conv2d:
        """The final convolution layer."""
        return [m for m in self.features.modules() if isinstance(m, nn.Conv2d)][-1]

    @property
    def gradcam_layer(self) -> nn.Module:
        """Output of the last convolution after its BatchNorm and ReLU (the Grad-CAM target)."""
        return [m for m in self.features.modules() if isinstance(m, nn.ReLU)][-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


def build_model(cfg: dict = CONFIG) -> SignCNN:
    """Create a SignCNN from the settings in `cfg`."""
    return SignCNN(
        num_classes=cfg["NUM_CLASSES"], filters=tuple(cfg["CONV_FILTERS"]), kernel_size=cfg["CONV_KERNEL"],
        conv_dropout=cfg["CONV_DROPOUT"], dense_units=cfg["DENSE_UNITS"], dense_dropout=cfg["DENSE_DROPOUT"],
        img_size=cfg["IMG_SIZE"],
    )


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
seed_everything(CONFIG["SEED"])
model = build_model().to(DEVICE)
print(model)
print(f"\nTrainable parameters: {count_trainable_parameters(model):,}")

# Trace the shape of a batch through the network.
with torch.no_grad():
    model.eval()
    h = X_train[:2]
    print(f"\n{'input':<12} {tuple(h.shape)}")
    for i, block in enumerate(model.features, start=1):
        h = block(h)
        print(f"{'block ' + str(i):<12} {tuple(h.shape)}")
    print(f"{'logits':<12} {tuple(model.classifier(h).shape)}")
assert next(model.parameters()).device.type == DEVICE.type

**Interpreting the output.** The shape trace confirms the arithmetic: 28 -> 14 -> 7 -> 4 pixels while channels grow 1 -> 32 -> 64 -> 128, and the output has one logit per letter. With about 0.8 million parameters this network is tiny by modern standards; it trains in about a second per epoch on a laptop GPU and runs comfortably in real time on a webcam stream. Most of the parameters sit in the first linear layer (2048 x 256 = 524k), which is why that layer gets the strongest dropout.

## 10. Training

**What.** Train the CNN with:

- **AdamW** (learning rate 1e-3, weight decay 1e-4). Adam adapts the step size per parameter; the "W" means weight decay is applied directly to the weights rather than mixed into the gradient, which regularises more predictably.
- **Cross-entropy with label smoothing 0.05.** Instead of a target of exactly 1 for the correct letter and 0 elsewhere, the target puts 0.95 + 0.05/24 on the correct letter and spreads the rest. This discourages extreme over-confidence, which helps with the threshold-based display live.
- **Batch size 256, up to 40 epochs**, with GPU augmentation of every training batch.
- **ReduceLROnPlateau** on validation loss: halve the learning rate after 3 epochs without improvement.
- **Early stopping**, written by hand: after 7 epochs without a validation-loss improvement, stop and restore the weights from the best epoch.
- **Mixed precision** on CUDA: `torch.autocast` runs convolutions and matrix multiplications in float16 (about twice as fast on tensor cores and half the memory), while `GradScaler` multiplies the loss by a large factor before backpropagation, so that small float16 gradients do not underflow to zero, then unscales them before the optimiser step.

**Train vs eval mode.** `model.train()` makes dropout active and batch norm use (and update) batch statistics. `model.eval()` turns dropout off and makes batch norm use its stored running statistics. Evaluation also runs under `torch.no_grad()`, so no computation graph is built, which saves memory and time.

**Two training accuracies.** The running accuracy during an epoch is measured on *augmented* batches with dropout on, so it is naturally lower than validation accuracy. For the over/underfitting check we therefore also measure accuracy on the *clean* training split in eval mode after each epoch (`train_acc_clean`), which compares like with like.

Loss and accuracy are accumulated on the GPU and only copied to the CPU once per epoch, because calling `.item()` on every batch forces the CPU to wait for the GPU each time.

In [ ]:
def train_one_epoch(model: nn.Module, X: torch.Tensor, y: torch.Tensor, optimizer: torch.optim.Optimizer,
                    criterion: nn.Module, scaler: torch.amp.GradScaler, cfg: dict = CONFIG,
                    augment: bool = True) -> tuple[float, float]:
    """One pass over (X, y) in shuffled mini-batches. Returns (mean loss, accuracy) on augmented batches."""
    model.train()
    total_loss = torch.zeros((), device=X.device)
    correct = torch.zeros((), device=X.device, dtype=torch.long)
    for idx in iterate_minibatches(len(X), cfg["BATCH_SIZE"], shuffle=True, device=X.device):
        xb, yb = X[idx], y[idx]
        if augment:
            xb = gpu_augment(xb, cfg)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()   # with AMP off, scale() and the scaler steps are pass-throughs
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.detach() * len(idx)
        correct += (logits.argmax(dim=1) == yb).sum()
    return total_loss.item() / len(X), correct.item() / len(X)


@torch.no_grad()
def evaluate(model: nn.Module, X: torch.Tensor, y: torch.Tensor, criterion: nn.Module,
             batch_size: int = CONFIG["EVAL_BATCH_SIZE"]) -> tuple[float, float]:
    """Mean loss and accuracy of `model` on (X, y), in eval mode and without gradients."""
    logits = predict_logits(model, X, batch_size)
    loss = criterion(logits, y).item()
    accuracy = (logits.argmax(dim=1) == y).float().mean().item()
    return loss, accuracy


def fit(model: nn.Module, X_tr: torch.Tensor, y_tr: torch.Tensor, X_va: torch.Tensor, y_va: torch.Tensor,
        lr: float, max_epochs: int, es_patience: int, cfg: dict = CONFIG,
        augment: bool = True) -> tuple[list[dict], int]:
    """Train with AdamW, ReduceLROnPlateau and hand-written early stopping on validation loss.

    The best state_dict (lowest validation loss) is kept on the device and restored at the end.
    Returns (history as a list of dicts, best epoch number).
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg["LR_FACTOR"], patience=cfg["LR_PATIENCE"], min_lr=cfg["MIN_LR"])
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["LABEL_SMOOTHING"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    history, best_loss, best_epoch, best_state, epochs_without_improvement = [], float("inf"), 0, None, 0
    if X_tr.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, max_epochs + 1):
        t0 = time.perf_counter()
        lr_now = optimizer.param_groups[0]["lr"]
        train_loss, train_acc = train_one_epoch(model, X_tr, y_tr, optimizer, criterion, scaler, cfg, augment)
        val_loss, val_acc = evaluate(model, X_va, y_va, criterion)
        clean_acc = evaluate(model, X_tr, y_tr, criterion)[1] if cfg["TRACK_CLEAN_TRAIN_ACC"] else float("nan")
        sync_device(X_tr.device)
        epoch_time = time.perf_counter() - t0
        peak_mb = torch.cuda.max_memory_allocated() / 2**20 if X_tr.device.type == "cuda" else float("nan")
        scheduler.step(val_loss)

        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                        "train_acc_clean": clean_acc, "val_loss": val_loss, "val_acc": val_acc,
                        "lr": lr_now, "epoch_time_s": epoch_time, "peak_gpu_mb": peak_mb})

        improved = val_loss < best_loss - cfg["EARLY_STOP_MIN_DELTA"]
        if improved:
            best_loss, best_epoch, epochs_without_improvement = val_loss, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        print(f"epoch {epoch:02d}/{max_epochs} | train loss {train_loss:.4f} acc {train_acc:.2%} "
              f"(clean {clean_acc:.2%}) | val loss {val_loss:.4f} acc {val_acc:.2%} | lr {lr_now:.1e} | "
              f"{epoch_time:.1f} s | peak {peak_mb:.0f} MB" + ("  *" if improved else ""))

        if epochs_without_improvement >= es_patience:
            print(f"Early stopping: no val-loss improvement for {es_patience} epochs.")
            break

    model.load_state_dict(best_state)
    print(f"Restored the weights from epoch {best_epoch} (val loss {best_loss:.4f}).")
    return history, best_epoch

In [ ]:
seed_everything(CONFIG["SEED"])
model = build_model().to(DEVICE)
t_start = time.perf_counter()
history, best_epoch = fit(model, X_train, y_train, X_val, y_val,
                          lr=CONFIG["LR"], max_epochs=CONFIG["MAX_EPOCHS"],
                          es_patience=CONFIG["EARLY_STOP_PATIENCE"], augment=CONFIG["AUGMENT"])
total_train_time = time.perf_counter() - t_start
if DEVICE.type == "cuda":
    print(f"Peak GPU memory during training: {torch.cuda.max_memory_allocated() / 2**20:.0f} MB")
    torch.cuda.empty_cache()
print(f"Total training time: {total_train_time:.0f} s over {len(history)} epochs")

history_df = pd.DataFrame(history).set_index("epoch")
history_df.tail()

**Interpreting the output.** Each line shows one epoch; a `*` marks a new best validation loss. In the reference run (RTX 3060 Laptop GPU, 6 GB):

- validation accuracy passed 98 % at epoch 2 and reached 100 % by epoch 5, helped by the near-duplicates; the clean training accuracy tracks it exactly;
- the running (augmented, dropout-on) training accuracy climbs more slowly, because augmented images are harder than clean ones;
- validation loss kept creeping down in small steps and the learning rate was halved several times by `ReduceLROnPlateau`. In one run training used all 40 epochs; in another, early stopping triggered at epoch 30 and restored epoch 23. That difference is the `cudnn.benchmark` effect described in section 2, and both runs reached the same test result. The loss levels off around 0.37 rather than 0: with label smoothing, the best achievable loss is above zero, because the target itself is not a hard 0/1 vector;
- the first epoch took about 7 s (CUDA start-up and cuDNN benchmarking the convolution algorithms), every later epoch about 1 s, and peak GPU memory was about 1.1 GB, mostly from the large evaluation batches and cuDNN workspace. A full run took under a minute.

## 11. Diagnosing the fit

**What.** Plot the learning curves and let a small function give a verdict:

- **Underfitting** if clean training accuracy (at the restored epoch) is below 95 %: the model has not even learned the training data.
- **Overfitting** if the train-validation accuracy gap is above 3 points, or the validation loss *rose steadily* after its minimum (at least 4 epochs after the minimum, at least 75 % of the epoch-to-epoch changes upward, and a total rise of at least 10 %).
- **Good fit** otherwise.

We also check that the validation loss was flat or still falling at the restored epoch, i.e. that the restored weights are not taken from a point where validation performance was already getting worse.

**Why.** Curves alone are easy to over-interpret; writing the rule down makes the judgement explicit and repeatable. Keep the caveat from section 6 in mind: because validation is drawn from the same near-duplicate pool as training, a small gap here does *not* prove the model generalises to new people or cameras. That question is answered by the test set and the robustness check.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
epochs = history_df.index

axes[0].plot(epochs, history_df["train_loss"], label="train (augmented)")
axes[0].plot(epochs, history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_ylabel("Cross-entropy (with label smoothing)")

axes[1].plot(epochs, history_df["train_acc"], label="train (augmented, dropout on)")
axes[1].plot(epochs, history_df["train_acc_clean"], label="train (clean, eval mode)")
axes[1].plot(epochs, history_df["val_acc"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_ylabel("Accuracy")

axes[2].plot(epochs, history_df["lr"], marker="o")
axes[2].set_yscale("log")
axes[2].set_title("Learning rate (ReduceLROnPlateau)")
axes[2].set_ylabel("Learning rate (log scale)")

for ax in axes:
    ax.axvline(best_epoch, color="grey", linestyle="--", linewidth=1, label=f"restored epoch {best_epoch}")
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def diagnose_fit(history_df: pd.DataFrame, best_epoch: int, cfg: dict = CONFIG) -> dict:
    """Classify the training run as underfitting, overfitting or a good fit, using the rules above."""
    best = history_df.loc[best_epoch]
    train_acc = best["train_acc_clean"] if not np.isnan(best["train_acc_clean"]) else best["train_acc"]
    gap = train_acc - best["val_acc"]

    val_loss = history_df["val_loss"].to_numpy()
    i_min = int(np.argmin(val_loss))
    after_min = val_loss[i_min:]
    steps = np.diff(after_min)
    rose_steadily = (
        len(steps) >= cfg["VAL_RISE_MIN_EPOCHS"]
        and np.mean(steps > 0) >= cfg["VAL_RISE_FRACTION"]
        and (after_min[-1] - after_min[0]) / after_min[0] >= cfg["VAL_RISE_REL"]
    )

    i_best = history_df.index.get_loc(best_epoch)
    window = val_loss[max(0, i_best - cfg["TREND_WINDOW"]):i_best + 1]
    flat_or_falling = window[-1] <= window.min() + cfg["EARLY_STOP_MIN_DELTA"]

    if train_acc < cfg["UNDERFIT_TRAIN_ACC"]:
        verdict = "UNDERFITTING"
        advice = ("The model has not learned the training data. Train longer (MAX_EPOCHS), raise LR, "
                  "reduce regularisation (CONV_DROPOUT, DENSE_DROPOUT, WEIGHT_DECAY), weaken augmentation "
                  "or make the network wider (CONV_FILTERS, DENSE_UNITS).")
    elif gap > cfg["OVERFIT_GAP"] or rose_steadily:
        verdict = "OVERFITTING"
        advice = ("The model fits the training data better than unseen data. Increase regularisation "
                  "(CONV_DROPOUT, DENSE_DROPOUT, WEIGHT_DECAY), strengthen augmentation, lower "
                  "EARLY_STOP_PATIENCE, or use a smaller network. Early stopping has already restored "
                  "the best weights, so the saved model is the least overfitted one from this run.")
    else:
        verdict = "GOOD FIT"
        advice = ("Training and validation agree and validation loss did not climb. Remember that the "
                  "random validation split is optimistic; the test set (section 12) is the real check.")

    result = {"verdict": verdict, "train_acc": train_acc, "val_acc": best["val_acc"], "gap": gap,
              "val_loss_rose_steadily": bool(rose_steadily), "val_loss_flat_or_falling_at_best": bool(flat_or_falling)}
    print(f"Restored epoch {best_epoch}: clean train acc {train_acc:.2%}, val acc {best['val_acc']:.2%}, "
          f"gap {gap * 100:+.2f} points (limit {cfg['OVERFIT_GAP'] * 100:.0f})")
    print(f"Validation loss rose steadily after its minimum: {rose_steadily}")
    print(f"Validation loss flat or still falling at the restored epoch: {flat_or_falling}")
    print(f"\nVERDICT: {verdict}\n{advice}")
    return result


fit_report = diagnose_fit(history_df, best_epoch)

**Interpreting the output.** In the reference run the verdict was **good fit**: clean training and validation accuracy were both 100 % (gap 0 points), and validation loss was still edging down at the restored epoch. A gap of zero at 100 % is itself a reminder of the near-duplicate problem: the validation split cannot distinguish between models that are all perfect on it, which is why the test set and the robustness check matter. If you change the settings and see **underfitting**, the model needs more capacity or less regularisation; if you see **overfitting**, it needs more regularisation or augmentation. The verdict is printed with the specific `CONFIG` keys to change, so the experiment loop is: change one setting, rerun from section 2, compare.

## 12. Final test evaluation (run once)

**What.** Evaluate the restored best model on the official test set: accuracy, macro F1, a per-letter classification report, a normalised confusion matrix, the most confused letter pairs, and examples of mistakes. The logistic-regression baseline is evaluated here too, for comparison.

**Why only once.** If we looked at test accuracy, changed a setting and looked again, the test set would gradually become part of the tuning process and its score would no longer be an unbiased estimate. Everything above was decided using the validation split only.

**Metrics.** Accuracy is the fraction of correct predictions. Macro F1 computes the F1 score (harmonic mean of precision and recall) for each letter and averages them with equal weight, so a single badly recognised letter pulls it down even if that letter is rare. The confusion matrix is **row-normalised**: each row (true letter) sums to 1, so cell (i, j) is the fraction of letter i predicted as letter j.

In [ ]:
test_logits = predict_logits(model, X_test)
test_probs = F.softmax(test_logits, dim=1)
test_conf, test_pred = test_probs.max(dim=1)

# Move to the CPU only now, for sklearn and plotting.
y_true = y_test.cpu().numpy()
y_pred = test_pred.cpu().numpy()
conf_np = test_conf.cpu().numpy()

test_acc = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(y_true, y_pred, average="macro")
baseline_test_acc = (predict_logits(baseline, X_test).argmax(1) == y_test).float().mean().item()

print(f"CNN test accuracy          : {test_acc:.2%}")
print(f"CNN test macro F1          : {test_macro_f1:.4f}")
print(f"Baseline (logistic) test acc: {baseline_test_acc:.2%}  (val was {baseline_val_acc:.2%})")
print(f"Errors: {(y_true != y_pred).sum()} of {len(y_true)}\n")
print(classification_report(y_true, y_pred, target_names=LETTERS, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(CONFIG["NUM_CLASSES"])))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=cm_norm >= 0.01, fmt="", cmap="Blues", vmin=0, vmax=1,
            xticklabels=LETTERS, yticklabels=LETTERS, cbar_kws={"label": "Fraction of true letter"}, ax=ax)
# Annotate only cells with at least 1 % so the plot stays readable.
for text in ax.texts:
    text.set_visible(False)
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        if cm_norm[i, j] >= 0.01:
            ax.text(j + 0.5, i + 0.5, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_title(f"Normalised confusion matrix on the test set (accuracy {test_acc:.2%})")
ax.set_xlabel("Predicted letter")
ax.set_ylabel("True letter")
plt.tight_layout()
plt.show()

# Hard confusion: how many images of letter i were predicted as j.
# Soft confusion: the average probability the model gave to letter j for images of letter i. This
# still ranks the "near misses" when there are few or no outright errors.
probs_np = test_probs.cpu().numpy()
n_cls = CONFIG["NUM_CLASSES"]
soft = np.stack([probs_np[y_true == i].mean(axis=0) for i in range(n_cls)])
pairs = [(LETTERS[i], LETTERS[j], int(cm[i, j]), cm_norm[i, j], soft[i, j])
         for i in range(n_cls) for j in range(n_cls) if i != j]
confused = (pd.DataFrame(pairs, columns=["true", "predicted", "errors", "fraction_of_true", "mean_prob_given"])
            .sort_values(["errors", "mean_prob_given"], ascending=False)
            .head(CONFIG["TOP_CONFUSED_PAIRS"]).reset_index(drop=True))
print(f"Top {CONFIG['TOP_CONFUSED_PAIRS']} confused (true -> predicted) letter pairs, "
      "ranked by error count, then by the mean probability given to the wrong letter:")
confused.style.format({"fraction_of_true": "{:.2%}", "mean_prob_given": "{:.4f}"})

In [ ]:
wrong = np.flatnonzero(y_true != y_pred)
# Probability the model gave to the *true* letter: low values are the hardest images.
p_true = probs_np[np.arange(len(y_true)), y_true]
if len(wrong) > 0:
    to_show = wrong[np.argsort(-conf_np[wrong])][:CONFIG["N_MISCLASSIFIED_SHOW"]]   # most confident mistakes first
    title = f"Misclassified test images ({len(to_show)} of {len(wrong)}): true letter, predicted letter, confidence"
else:
    to_show = np.argsort(p_true)[:CONFIG["N_MISCLASSIFIED_SHOW"]]
    title = "No misclassified test images - showing the least confident correct predictions instead"
print(title)

cols = CONFIG["GRID_COLS"]
rows = int(np.ceil(len(to_show) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2.3 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, to_show):
    ax.imshow(test_images[i], cmap="gray", vmin=0, vmax=CONFIG["PIXEL_MAX"])
    ax.set_title(f"true {LETTERS[y_true[i]]} / pred {LETTERS[y_pred[i]]}\n{conf_np[i]:.0%}", fontsize=9,
                 color="red" if y_true[i] != y_pred[i] else "black")
fig.suptitle(title)
plt.tight_layout()
plt.show()

In [ ]:
def check_targets(test_acc: float, test_macro_f1: float, fit_report: dict, cfg: dict = CONFIG) -> None:
    """Print a plain PASS/MISS line for each project target, with a suggestion when one is missed."""
    checks = [
        ("Test accuracy", test_acc >= cfg["TARGET_TEST_ACC"], f"{test_acc:.2%} (target >= {cfg['TARGET_TEST_ACC']:.0%})",
         "strengthen augmentation (AUG_ROTATION_DEG, AUG_TRANSLATE_FRAC, AUG_BRIGHTNESS), train longer "
         "(MAX_EPOCHS, EARLY_STOP_PATIENCE) or widen the network (CONV_FILTERS, DENSE_UNITS)"),
        ("Macro F1", test_macro_f1 >= cfg["TARGET_MACRO_F1"], f"{test_macro_f1:.4f} (target >= {cfg['TARGET_MACRO_F1']})",
         "look at the confused pairs above; more geometric augmentation usually helps M/N/S/T/A/E"),
        ("Train-val gap", fit_report["gap"] <= cfg["OVERFIT_GAP"], f"{fit_report['gap'] * 100:+.2f} points (target <= {cfg['OVERFIT_GAP'] * 100:.0f})",
         "increase CONV_DROPOUT / DENSE_DROPOUT / WEIGHT_DECAY or augmentation strength"),
        ("Val loss flat/falling at restored epoch", fit_report["val_loss_flat_or_falling_at_best"],
         "yes" if fit_report["val_loss_flat_or_falling_at_best"] else "no",
         "lower EARLY_STOP_PATIENCE or LR_PATIENCE"),
    ]
    for name, ok, value, suggestion in checks:
        print(f"[{'PASS' if ok else 'MISS'}] {name}: {value}")
        if not ok:
            print(f"        -> target missed. Suggested change: {suggestion}. "
                  "Tune on the validation split only, never on the test set.")


check_targets(test_acc, test_macro_f1, fit_report)

**Interpreting the output.**

- In the reference run the CNN classified **all 7,172 test images correctly** (accuracy 100.00 %, macro F1 1.000), above the 98 % / 0.98 targets. Because `cudnn.benchmark` makes runs differ slightly, expect anywhere from about 99.5 % to 100 % on a rerun. The logistic-regression baseline drops from about 80 % on validation to about 63 % on test, which shows both how optimistic the near-duplicate validation split is and how much the CNN's spatial features and augmentation add.
- With no outright errors, the confused-pairs table is ranked by the *mean probability given to the wrong letter*, which shows the near misses. In the reference run the top pair was **Y -> I** (both raise the little finger; Y also extends the thumb), followed by pairs such as R -> S and T -> X. These probabilities are tiny (below 0.3 %), but they show which shapes the model finds most alike.
- In other runs, and above all live, the classic confusions are the fist-based letters **M, N, S, T, A and E**, which differ only in where the thumb sits over or beside a closed fist. At 28 x 28 pixels the thumb is only a few pixels wide, so expect these to be the hardest letters in front of the webcam.
- The grid shows the misclassified images, or the least confident correct ones when there are no errors. These tend to be darker, more cluttered or unusually framed images, a hint of what can go wrong with webcam crops.
- **A perfect test score does not mean the problem is solved.** The test images come from the same kind of photo sessions as the training images (similar backgrounds, lighting and framing). The live webcam is a different distribution, so accuracy there will be lower. Sections 13, 16 and 17 deal with that gap.

## 13. Robustness check

**What.** Re-evaluate the test set after deliberately degrading it, all on the GPU:

- **Brightness shifts:** $x' = \operatorname{clamp}(x + \delta, 0, 1)$ for several $\delta$.
- **Gaussian blur:** convolution with a Gaussian kernel $G(u) = \frac{1}{Z}\exp\!\left(-\frac{u^2}{2\sigma^2}\right)$, built by hand. A 2-D Gaussian is *separable* ($G(u, v) = G(u)\,G(v)$), so we blur rows with a 1-D kernel and then columns, which costs $2k$ instead of $k^2$ multiplications per pixel for a kernel of width $k$. Borders are handled by reflecting the image.
- **Rotation:** the affine warp from section 7 with a fixed angle, including angles beyond the ±10° used in training.
- **Gaussian noise** of increasing standard deviation.

**Why.** Webcam crops differ from the dataset in exactly these ways: lighting, motion blur and focus, the angle you hold your hand at, and sensor noise in dim rooms. A model that collapses under mild versions of these will be fragile live. The same degradations are applied to the validation split as well. **If you want to act on these results** (for example by widening an augmentation range), base the decision on the validation column and keep the test column for reporting; otherwise the test set would quietly become part of the tuning process.

In [ ]:
def adjust_brightness(x: torch.Tensor, delta: float) -> torch.Tensor:
    return (x + delta).clamp(0.0, 1.0)


def gaussian_kernel_1d(sigma: float, radius_sigmas: int, device: torch.device) -> torch.Tensor:
    """Normalised 1-D Gaussian kernel of half-width ceil(radius_sigmas * sigma)."""
    radius = max(1, int(np.ceil(radius_sigmas * sigma)))
    u = torch.arange(-radius, radius + 1, device=device, dtype=torch.float32)
    kernel = torch.exp(-(u ** 2) / (2 * sigma ** 2))
    return kernel / kernel.sum()


def gaussian_blur(x: torch.Tensor, sigma: float, radius_sigmas: int = CONFIG["BLUR_RADIUS_SIGMAS"]) -> torch.Tensor:
    """Separable Gaussian blur of a batch (N, 1, H, W): a horizontal pass, then a vertical pass."""
    k = gaussian_kernel_1d(sigma, radius_sigmas, x.device)
    r = len(k) // 2
    x = F.conv2d(F.pad(x, (r, r, 0, 0), mode="reflect"), k.view(1, 1, 1, -1))   # along rows
    x = F.conv2d(F.pad(x, (0, 0, r, r), mode="reflect"), k.view(1, 1, -1, 1))   # along columns
    return x


def rotate(x: torch.Tensor, degrees: float) -> torch.Tensor:
    """Rotate every image in the batch by the same angle (no zoom, no shift)."""
    n = x.shape[0]
    angles = torch.full((n,), float(np.deg2rad(degrees)), device=x.device)
    return apply_affine(x, build_affine_matrices(angles, torch.ones(n, device=x.device), torch.zeros(n, 2, device=x.device)))


def add_noise(x: torch.Tensor, std: float, seed: int = CONFIG["SEED"]) -> torch.Tensor:
    generator = torch.Generator(device=x.device).manual_seed(seed)   # same noise every run
    noise = torch.randn(x.shape, generator=generator, device=x.device)
    return (x + std * noise).clamp(0.0, 1.0)


conditions = [("clean", lambda x: x)]
conditions += [(f"brightness {d:+.2f}", lambda x, d=d: adjust_brightness(x, d)) for d in CONFIG["ROBUST_BRIGHTNESS"]]
conditions += [(f"blur sigma {s}", lambda x, s=s: gaussian_blur(x, s)) for s in CONFIG["ROBUST_BLUR_SIGMA"]]
conditions += [(f"rotate {a:+.1f} deg", lambda x, a=a: rotate(x, a)) for a in CONFIG["ROBUST_ROTATION_DEG"]]
conditions += [(f"noise std {s}", lambda x, s=s: add_noise(x, s)) for s in CONFIG["ROBUST_NOISE_STD"]]

robust_rows = []
with torch.no_grad():
    for name, transform in conditions:
        row = {"condition": name}
        for split, X_eval, y_eval in [("val_accuracy", X_val, y_val), ("accuracy", X_test, y_test)]:
            row[split] = (predict_logits(model, transform(X_eval)).argmax(1) == y_eval).float().mean().item()
        robust_rows.append(row)
robust_df = pd.DataFrame(robust_rows)[["condition", "val_accuracy", "accuracy"]]

fig, ax = plt.subplots(figsize=(10, 7))
ypos = np.arange(len(robust_df))
bar_h = 0.4
ax.barh(ypos - bar_h / 2, robust_df["val_accuracy"] * 100, height=bar_h, color="tab:orange", label="validation")
ax.barh(ypos + bar_h / 2, robust_df["accuracy"] * 100, height=bar_h, color="tab:blue", label="test")
ax.set_yticks(ypos, robust_df["condition"])
ax.axvline(CONFIG["TARGET_TEST_ACC"] * 100, color="red", linestyle="--", label="98 % target")
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_title("Accuracy under image degradations (validation and test)")
ax.set_xlabel("Test accuracy (%)")
ax.set_ylabel("Condition")
ax.legend()
plt.tight_layout()
plt.show()

# What do the degradations look like on one test image?
fig, axes = plt.subplots(2, int(np.ceil(len(conditions) / 2)), figsize=(16, 4.5))
for ax in axes.flat:
    ax.axis("off")
sample = X_test[:1]
for ax, (name, transform) in zip(axes.flat, conditions):
    ax.imshow(transform(sample)[0, 0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(name, fontsize=8)
fig.suptitle("The degradations applied to one test image")
plt.tight_layout()
plt.show()
robust_df.rename(columns={"accuracy": "test_accuracy"}).style.format({"val_accuracy": "{:.2%}", "test_accuracy": "{:.2%}"})

**Interpreting the output.** In the reference run:

- **Brightness** shifts of ±0.15 cost nothing and ±0.30 at most about 1 point. Contrast and brightness jitter in training cover this.
- **Rotation** up to ±15° (beyond the ±10° training range) cost almost nothing: the model tolerates a tilted hand well.
- **Blur** is fine up to σ = 1 pixel, but at σ = 1.5 accuracy falls to roughly 72-75 %. At that level the finger gaps, which are only 1-2 pixels wide at 28 x 28, are smoothed away.
- **Noise** is the clear weakness: fine at σ = 0.05, about 55-58 % at σ = 0.10 and near chance at σ = 0.20. Training only used light noise (σ = 0.02), and the network clearly relies on fine texture and edges that strong noise destroys.

How much does this matter live? Less than it seems, thanks to the preprocessing. The webcam crop is around 250-350 pixels wide and is shrunk to 28 x 28 with `INTER_AREA`, so each model pixel is the **average of roughly 100 camera pixels**. Averaging $n$ independent noisy values divides the noise standard deviation by $\sqrt{n}$, about 10 here, so heavy sensor noise of σ = 0.1 becomes about 0.01 after resizing. Blur shrinks by the same factor of about 10: a 15-pixel motion blur in the camera image becomes about 1.5 pixels at 28 x 28. That level does hurt, so **keep your hand still** while signing. Validation and test agree closely in every row, so the validation column is a safe basis for any changes. To harden the model you could widen `AUG_NOISE_STD` or add a random-blur augmentation, retrain, and compare the *validation* column.

## 14. Grad-CAM: where is the model looking?

**What.** Gradient-weighted Class Activation Mapping (Grad-CAM, Selvaraju et al., 2017) highlights the image regions that most increased the score of a chosen class. It is implemented here with two hooks on the last convolution layer, attached at its output *after* batch norm and ReLU:

- a **forward hook** stores the feature maps $A^k$ (128 maps of 7 x 7);
- a **backward hook** stores the gradient of the class score $y^c$ (the logit, before softmax) with respect to those maps, $\partial y^c / \partial A^k$.

Each feature map gets an importance weight, the spatial average of its gradient, and the heatmap is the ReLU of the weighted sum:

$$
\alpha_k^c = \frac{1}{Z}\sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}, \qquad
L^c = \operatorname{ReLU}\!\Big(\sum_k \alpha_k^c A^k\Big).
$$

The ReLU keeps only regions that push *towards* class $c$. We hook the rectified output (conv -> BN -> ReLU) rather than the raw convolution because the raw output carries arbitrary per-channel offsets that the following batch norm removes; summing those offsets produces heat that has nothing to do with the decision. The original Grad-CAM paper likewise uses rectified convolutional feature maps. The 7 x 7 map is upsampled to 28 x 28 with bilinear interpolation (the same interpolation as in section 7), scaled to $[0, 1]$ and overlaid on the image.

**Why.** High accuracy can hide a model that relies on the wrong cues, for example background texture or the edge of the frame. If the heatmaps sit on the fingers and thumb, we have some evidence the model has learned hand shape, which is what has to transfer to the webcam.

In [ ]:
class GradCAM:
    """Grad-CAM using a forward hook (activations) and a full backward hook (gradients) on one layer."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        self._handles = [
            target_layer.register_forward_hook(self._save_activations),
            target_layer.register_full_backward_hook(self._save_gradients),
        ]

    def _save_activations(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x: torch.Tensor, class_idx: int | None = None) -> tuple[np.ndarray, int, float]:
        """Return (heatmap 28 x 28 in [0, 1], class used, probability of that class) for one image."""
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)                       # full precision; gradients are needed here
        probs = F.softmax(logits, dim=1)
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        logits[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)            # alpha_k: (1, C, 1, 1)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))  # (1, 1, 7, 7)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam / (cam.max() + torch.finfo(cam.dtype).eps)
        return cam[0, 0].cpu().numpy(), class_idx, probs[0, class_idx].item()

    def remove(self) -> None:
        for handle in self._handles:
            handle.remove()


cam_engine = GradCAM(model, model.gradcam_layer)
right = np.flatnonzero(y_true == y_pred)
rng = np.random.default_rng(CONFIG["SEED"])
picks = list(rng.choice(right, CONFIG["GRADCAM_N_CORRECT"], replace=False))
hardest = np.argsort(p_true)[:CONFIG["GRADCAM_N_WRONG"]]   # lowest probability for the true letter (errors first)
picks += list(hardest)

fig, axes = plt.subplots(3, len(picks), figsize=(2.1 * len(picks), 6.8), squeeze=False)
for col, i in enumerate(picks):
    x = X_test[i:i + 1]
    heatmap, cls, prob = cam_engine(x)
    image = x[0, 0].cpu().numpy()
    axes[0, col].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(f"true {LETTERS[y_true[i]]}", fontsize=9)
    axes[1, col].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    axes[1, col].set_title(f"CAM for {LETTERS[cls]}", fontsize=9)
    axes[2, col].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[2, col].imshow(heatmap, cmap="jet", alpha=CONFIG["GRADCAM_ALPHA"], vmin=0, vmax=1)
    axes[2, col].set_title(f"pred {LETTERS[cls]} ({prob:.0%})", fontsize=9,
                           color="green" if cls == y_true[i] else "red")
    for row in range(3):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
for row, label in enumerate(["input", "Grad-CAM", "overlay"]):
    axes[row, 0].set_ylabel(label)
fig.suptitle(f"Grad-CAM on the last conv layer: {CONFIG['GRADCAM_N_CORRECT']} random correct images (left) "
             f"and the {CONFIG['GRADCAM_N_WRONG']} hardest test images (right; red = misclassified)")
plt.tight_layout()
plt.show()
cam_engine.remove()   # always remove hooks when done, or they keep firing on every forward pass

**Interpreting the output.** In the reference run the heat sits on the hand: on the palm and finger region for B and V, on the extended fingers for H and G, and on the raised fingers for Y. The model is using hand shape, not the background or the image border. For the hardest images the heat is sometimes spread wider or shifted (for the T on the far right it reaches the hand's edge and the background), which fits their lower confidence. If an error does occur, look for heat on the wrong part of the hand, for example ignoring the thumb in an M/N/S/T confusion.

The heatmap has only 7 x 7 cells of real resolution before upsampling, so it shows *regions*, not precise pixels. Grad-CAM also caught a real bug during development: with the default floor-mode pooling, the last row and column of the 7 x 7 map never reached the classifier, and the heatmaps lit up along the bottom and right edges. Section 9 explains the `ceil_mode=True` fix.

## 15. Saving and loading the model

**What.** Save three files to `artifacts/`:

- `sign_cnn_best.pt`: the restored best `state_dict` (weights only, not the Python class);
- `label_map.json`: class index -> letter, and class index -> raw dataset label;
- `config.json`: the full `CONFIG`, so the architecture and preprocessing can be rebuilt exactly.

`load_model()` rebuilds the architecture from the saved config, loads the weights with `map_location` set to the current device (so a model trained on CUDA also loads on a CPU-only or Apple machine), and switches to `eval()` mode.

**Why a state_dict.** Saving the whole model object with pickle ties the file to the exact code and file layout that created it. A state_dict is just named tensors, and `torch.load(..., weights_only=True)` refuses to execute arbitrary code from the file.

**Running the webcam demo on its own.** After a kernel restart you do not need to retrain or even load the dataset. Run, in order: the code cells of sections 2 and 3, the first code cell of section 5 (preprocessing functions), the first code cell of section 9 (model classes), the second code cell of this section (`load_model`), and then section 16.

In [ ]:
OUTPUT_DIR = Path(CONFIG["OUTPUT_DIR"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / CONFIG["MODEL_FILE"]
LABEL_MAP_PATH = OUTPUT_DIR / CONFIG["LABEL_MAP_FILE"]
CONFIG_PATH = OUTPUT_DIR / CONFIG["CONFIG_FILE"]

torch.save(model.state_dict(), MODEL_PATH)
with open(LABEL_MAP_PATH, "w") as f:
    json.dump({"idx_to_letter": IDX_TO_LETTER,
               "idx_to_raw_label": {i: int(raw) for i, raw in enumerate(RAW_LABELS)}}, f, indent=2)
with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
for path in (MODEL_PATH, LABEL_MAP_PATH, CONFIG_PATH):
    print(f"saved {path} ({path.stat().st_size / 1024:.1f} KB)")

In [ ]:
OUTPUT_DIR = Path(CONFIG["OUTPUT_DIR"])
MODEL_PATH = OUTPUT_DIR / CONFIG["MODEL_FILE"]
LABEL_MAP_PATH = OUTPUT_DIR / CONFIG["LABEL_MAP_FILE"]
CONFIG_PATH = OUTPUT_DIR / CONFIG["CONFIG_FILE"]


def load_label_map(path: Path = LABEL_MAP_PATH) -> dict[int, str]:
    """Read the class index -> letter mapping (JSON keys are strings, so convert back to int)."""
    with open(path) as f:
        return {int(k): v for k, v in json.load(f)["idx_to_letter"].items()}


def load_model(model_path: Path = MODEL_PATH, config_path: Path = CONFIG_PATH,
               device: torch.device = DEVICE) -> SignCNN:
    """Rebuild the CNN from the saved config, load the weights onto `device` and switch to eval mode."""
    with open(config_path) as f:
        saved_cfg = json.load(f)
    net = build_model(saved_cfg).to(device)
    state = torch.load(model_path, map_location=device, weights_only=True)
    net.load_state_dict(state)
    net.eval()
    return net


IDX_TO_LETTER = load_label_map()
LETTERS = [IDX_TO_LETTER[i] for i in range(len(IDX_TO_LETTER))]
LETTER_TO_IDX = {letter: i for i, letter in IDX_TO_LETTER.items()}
live_model = load_model()
print(f"Loaded {MODEL_PATH} onto {next(live_model.parameters()).device}, training mode = {live_model.training}")

# Check the reloaded model gives the same predictions as the one in memory (on validation data).
if "model" in globals() and "X_val" in globals():
    same = torch.equal(predict_logits(live_model, X_val).argmax(1), predict_logits(model, X_val).argmax(1))
    print(f"Reloaded model agrees with the trained model on every validation image: {same}")
    assert same

**Interpreting the output.** The weights file is only about 3 MB. The reloaded model lives on the same device, is in eval mode, and makes exactly the same predictions as the model in memory, so the live demo below uses exactly the model that was evaluated above.

## 16. Live webcam demo

This section turns a webcam frame into a letter. It is split into small, testable pieces:

1. **`HandDetector`** wraps MediaPipe and returns 21 landmark positions in pixels. Nothing else depends on which MediaPipe API is installed.
2. **Geometry and preprocessing:** build a square, padded box from the landmarks, crop it, and convert it to the 28 x 28 model input.
3. **Smoothing and word building:** average the last 10 probability vectors, apply the 0.70 threshold, and type a letter that is held for about a second.
4. **Drawing:** landmarks, box, the enlarged model-input inset and the information panel.
5. **The capture loop**, which opens the camera and ties everything together.

### 16.1 Hand detection with MediaPipe

MediaPipe's hand tracker first runs a palm detector, then a landmark model that predicts 21 points on the hand (wrist, and four points along each finger). In video mode it reuses the previous frame's hand position, which is faster and steadier than detecting from scratch each frame; the detection confidence (0.6) controls when a new hand is accepted and the tracking confidence (0.5) when tracking is considered lost.

Newer MediaPipe releases provide the **Tasks** API (`HandLandmarker`, which needs a model file, `hand_landmarker.task`, downloaded once); older ones provide the legacy `mp.solutions.hands`, and some recent releases have removed it. The class below uses whichever is available.

In [ ]:
# The 21 MediaPipe hand landmarks and the bones that connect them (0 = wrist; 4, 8, 12, 16, 20 = fingertips).
HAND_CONNECTIONS = (
    (0, 1), (1, 2), (2, 3), (3, 4),            # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),            # index finger
    (5, 9), (9, 10), (10, 11), (11, 12),       # middle finger
    (9, 13), (13, 14), (14, 15), (15, 16),     # ring finger
    (13, 17), (17, 18), (18, 19), (19, 20),    # little finger
    (0, 17),                                    # palm edge
)


class HandDetector:
    """Detect hands with whichever MediaPipe API is installed and return landmarks in pixel coordinates.

    detect(frame_bgr) -> list of (21, 2) float arrays of (x, y) pixel positions, one per hand.
    """

    def __init__(self, cfg: dict = CONFIG, video_mode: bool = True):
        self.video_mode = video_mode
        self._last_timestamp_ms = -1
        if HAS_MP_TASKS:
            model_path = Path(cfg["HAND_MODEL_PATH"])
            if not model_path.exists():
                print(f"Downloading the MediaPipe hand model to {model_path} ...")
                urllib.request.urlretrieve(cfg["HAND_MODEL_URL"], model_path)
            options = mp_vision.HandLandmarkerOptions(
                base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
                running_mode=mp_vision.RunningMode.VIDEO if video_mode else mp_vision.RunningMode.IMAGE,
                num_hands=cfg["MAX_HANDS"],
                min_hand_detection_confidence=cfg["HAND_DETECTION_CONF"],
                min_hand_presence_confidence=cfg["HAND_TRACKING_CONF"],
                min_tracking_confidence=cfg["HAND_TRACKING_CONF"],
            )
            self._detector = mp_vision.HandLandmarker.create_from_options(options)
            self.backend = "MediaPipe Tasks HandLandmarker"
        elif HAS_MP_SOLUTIONS:
            self._detector = mp.solutions.hands.Hands(
                static_image_mode=not video_mode,
                max_num_hands=cfg["MAX_HANDS"],
                min_detection_confidence=cfg["HAND_DETECTION_CONF"],
                min_tracking_confidence=cfg["HAND_TRACKING_CONF"],
            )
            self.backend = "MediaPipe legacy solutions.hands"
        else:
            raise RuntimeError("This MediaPipe installation has neither the Tasks API nor mp.solutions.hands.")

    def _next_timestamp_ms(self) -> int:
        """Video mode needs strictly increasing timestamps."""
        now = int(time.monotonic() * 1000)
        self._last_timestamp_ms = max(now, self._last_timestamp_ms + 1)
        return self._last_timestamp_ms

    def detect(self, frame_bgr: np.ndarray) -> list[np.ndarray]:
        height, width = frame_bgr.shape[:2]
        rgb = np.ascontiguousarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))   # MediaPipe expects RGB
        if self.backend.startswith("MediaPipe Tasks"):
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            if self.video_mode:
                result = self._detector.detect_for_video(image, self._next_timestamp_ms())
            else:
                result = self._detector.detect(image)
            hands = result.hand_landmarks
        else:
            result = self._detector.process(rgb)
            hands = [h.landmark for h in (result.multi_hand_landmarks or [])]
        # Landmarks are normalised to [0, 1]; convert to pixel coordinates.
        return [np.array([[lm.x * width, lm.y * height] for lm in hand], dtype=np.float32) for hand in hands]

    def close(self) -> None:
        self._detector.close()


_probe = HandDetector(video_mode=False)
print(f"Hand detector backend: {_probe.backend}")
_probe.close()

**Interpreting the output.** The line tells you which MediaPipe API is in use. With MediaPipe 1.2.x the legacy `mp.solutions` module no longer exists, so the Tasks API is used. The first time, it downloads `hand_landmarker.task` (about 7.5 MB) next to the notebook; after that it loads from disk. The `INFO: ... LiteRT` / `XNNPACK` lines that MediaPipe prints come from its C++ runtime and are harmless.

### 16.2 From landmarks to model input

**Square box.** Take the smallest rectangle containing all 21 landmarks, then use its longer side $L$ as the side of a square centred on the rectangle's centre, enlarged by the padding: side $= L(1 + 0.25)$. A square matters because the crop is later resized to 28 x 28: resizing a non-square crop would stretch the hand and change its shape. The padding keeps the fingertips and the edges of the hand inside the crop, as in the dataset images, where the hand nearly fills the frame. Landmarks mark joint centres, not the skin outline, so without padding the fingertips would be cut off.

**Clamping.** If the square sticks out of the frame, it is first *shifted* back inside, and only shrunk if it is larger than the frame itself. Shifting rather than cutting keeps the crop square.

**Preprocessing, matching training.** Grayscale conversion uses the standard luminance weights $Y = 0.299R + 0.587G + 0.114B$ (what `cv2.COLOR_BGR2GRAY` computes, and close to how the dataset images were made). Optional **CLAHE** (Contrast-Limited Adaptive Histogram Equalisation) equalises the histogram separately in small tiles, with the clip limit capping how much any intensity can be amplified so that noise is not blown up; it helps in uneven or dim lighting. The crop is then shrunk to 28 x 28 with `INTER_AREA` (area averaging, which avoids aliasing when downsampling by a large factor) and scaled by `preprocess_for_model`.

**Mirroring.** The display is mirrored so the window behaves like a mirror. The dataset photos were taken by a camera facing the signer, i.e. *not* mirrored, so by default (`MIRROR_INPUT = False`) the crop is flipped back before it reaches the model. If you sign with the other hand from the dataset signers, mirroring the input may work better; press **m** to compare.

In [ ]:
def landmarks_to_square_box(points: np.ndarray, frame_shape: tuple, padding: float = CONFIG["BOX_PADDING"]) -> tuple[int, int, int, int]:
    """Square, padded box (x0, y0, x1, y1) around the landmarks, kept inside the frame."""
    frame_h, frame_w = frame_shape[:2]
    x_min, y_min = points.min(axis=0)
    x_max, y_max = points.max(axis=0)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(x_max - x_min, y_max - y_min) * (1 + padding)
    side = int(round(min(side, frame_w, frame_h)))          # cannot be larger than the frame
    x0 = int(round(cx - side / 2))
    y0 = int(round(cy - side / 2))
    x0 = min(max(x0, 0), frame_w - side)                    # shift (not cut) back inside the frame
    y0 = min(max(y0, 0), frame_h - side)
    return x0, y0, x0 + side, y0 + side


def crop_to_model_image(frame_bgr: np.ndarray, box: tuple[int, int, int, int], mirror_input: bool,
                        clahe: cv2.CLAHE | None) -> np.ndarray:
    """Crop the box and apply the same preprocessing as the dataset: gray -> (CLAHE) -> 28 x 28 uint8.

    `frame_bgr` is the mirrored display frame; unless mirror_input is True the crop is flipped
    back so the model sees the camera's real (un-mirrored) view, like the dataset photos.
    """
    x0, y0, x1, y1 = box
    crop = frame_bgr[y0:y1, x0:x1]
    if not mirror_input:
        crop = cv2.flip(crop, 1)
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    if clahe is not None:
        gray = clahe.apply(gray)
    size = CONFIG["IMG_SIZE"]
    return cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)


def predict_probabilities(net: nn.Module, image_28: np.ndarray, device: torch.device = DEVICE) -> np.ndarray:
    """Softmax probabilities for one 28 x 28 uint8 image. Only the probability vector returns to the CPU."""
    x = preprocess_for_model(image_28, device)
    with torch.inference_mode():
        probs = F.softmax(net(x), dim=1)[0]
    return probs.float().cpu().numpy()


class PredictionSmoother:
    """Average the last `window` probability vectors and apply a confidence threshold."""

    def __init__(self, window: int = CONFIG["SMOOTHING_WINDOW"], threshold: float = CONFIG["CONF_THRESHOLD"]):
        self.buffer = deque(maxlen=window)
        self.threshold = threshold

    def add(self, probs: np.ndarray) -> None:
        self.buffer.append(probs)

    def clear(self) -> None:
        self.buffer.clear()

    def result(self, top_k: int = CONFIG["TOP_K"]) -> tuple[str | None, float, list[tuple[str, float]]]:
        """(letter or "?" or None if empty, smoothed confidence, top-k [(letter, prob)])."""
        if not self.buffer:
            return None, 0.0, []
        mean = np.mean(self.buffer, axis=0)
        order = np.argsort(mean)[::-1][:top_k]
        top = [(IDX_TO_LETTER[int(i)], float(mean[i])) for i in order]
        letter, confidence = top[0]
        return (letter if confidence >= self.threshold else "?"), confidence, top


class WordBuilder:
    """Type a letter when it has been held steadily for `hold_seconds`; one letter per hold."""

    def __init__(self, hold_seconds: float = CONFIG["HOLD_SECONDS"]):
        self.hold_seconds = hold_seconds
        self.text = ""
        self._candidate = None
        self._since = 0.0
        self._typed = False

    def update(self, letter: str | None, now: float) -> float:
        """Feed the current smoothed letter; returns hold progress in [0, 1] for display."""
        if letter is None or letter == "?":
            self._candidate, self._typed = None, False
            return 0.0
        if letter != self._candidate:
            self._candidate, self._since, self._typed = letter, now, False
        held = now - self._since
        if not self._typed and held >= self.hold_seconds:
            self.text += letter
            self._typed = True            # change letter (or drop the hand) to type the same letter again
        return 1.0 if self._typed else held / self.hold_seconds

    def space(self) -> None:
        self.text += " "

    def backspace(self) -> None:
        self.text = self.text[:-1]

    def clear(self) -> None:
        self.text = ""

### 16.3 Drawing the overlay and the information panel

The window has two parts: the (mirrored) video frame with the landmarks, the crop box and the enlarged model input in the top-right corner, and a black panel underneath with the letter, its confidence, the top-3 alternatives, the typed text, FPS and the device.

The **inset** is the most useful debugging tool in the demo. It shows the exact 28 x 28 image the CNN receives, enlarged with nearest-neighbour interpolation so the real pixels stay visible. If predictions are wrong, compare it with the dataset samples from section 4: is the hand centred and filling the square? Is it too dark? Is the background busy? Is it mirrored?

In [ ]:
GREEN, RED, WHITE, GREY, YELLOW, CYAN = (0, 200, 0), (0, 0, 255), (255, 255, 255), (160, 160, 160), (0, 220, 255), (255, 200, 0)
FONT = cv2.FONT_HERSHEY_SIMPLEX


def draw_landmarks(frame: np.ndarray, points: np.ndarray) -> None:
    """Draw the 21 landmarks and the connecting bones in place."""
    pts = points.astype(int)
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, tuple(pts[a]), tuple(pts[b]), WHITE, 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 4, RED, -1, cv2.LINE_AA)


def draw_inset(frame: np.ndarray, image_28: np.ndarray, cfg: dict = CONFIG) -> None:
    """Paste the enlarged 28 x 28 model input into the top-right corner of the frame, in place."""
    size, margin = cfg["INSET_SIZE"], cfg["INSET_MARGIN"]
    big = cv2.resize(image_28, (size, size), interpolation=cv2.INTER_NEAREST)   # keep the pixels visible
    x0 = frame.shape[1] - size - margin
    frame[margin:margin + size, x0:x0 + size] = cv2.cvtColor(big, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(frame, (x0 - 1, margin - 1), (x0 + size, margin + size), YELLOW, 1)
    cv2.putText(frame, "model input 28x28", (x0, margin + size + 16), FONT, 0.45, YELLOW, 1, cv2.LINE_AA)


def render_panel(width: int, letter: str | None, confidence: float, top: list, text: str, hold: float,
                 fps: float, device_name: str, mirror_input: bool, use_clahe: bool,
                 cfg: dict = CONFIG) -> np.ndarray:
    """Black information panel shown under the video."""
    panel = np.zeros((cfg["PANEL_HEIGHT"], width, 3), dtype=np.uint8)
    if letter is None:
        cv2.putText(panel, "No hand detected", (20, 70), FONT, 1.2, GREY, 2, cv2.LINE_AA)
    else:
        colour = GREEN if letter != "?" else YELLOW
        cv2.putText(panel, letter, (20, 95), FONT, 3.2, colour, 6, cv2.LINE_AA)
        cv2.putText(panel, f"{confidence:.0%}", (120, 60), FONT, 1.1, colour, 2, cv2.LINE_AA)
        bar_w = 150
        cv2.rectangle(panel, (120, 75), (120 + bar_w, 87), GREY, 1)              # hold-to-type progress
        cv2.rectangle(panel, (120, 75), (120 + int(bar_w * hold), 87), colour, -1)
        alternatives = "   ".join(f"{l} {p:.0%}" for l, p in top)
        cv2.putText(panel, f"top {len(top)}: {alternatives}", (300, 45), FONT, 0.65, WHITE, 1, cv2.LINE_AA)
    cv2.putText(panel, f"text: {text[-40:]}_", (20, 140), FONT, 0.8, CYAN, 2, cv2.LINE_AA)
    status = (f"FPS {fps:4.1f} | device {device_name} | mirror input (m): {'on' if mirror_input else 'off'}"
              f" | CLAHE (h): {'on' if use_clahe else 'off'}")
    cv2.putText(panel, status, (20, 172), FONT, 0.48, GREY, 1, cv2.LINE_AA)
    cv2.putText(panel, "q quit  space  backspace  c clear", (300, 80), FONT, 0.5, GREY, 1, cv2.LINE_AA)
    return panel

In [ ]:
# Dry run without a camera: push a test image through the same crop -> preprocess -> predict path
# and render the panel, to check everything works before opening the webcam.
def dry_run_preview(net: nn.Module, image: np.ndarray, true_letter: str) -> None:
    """Push one dataset image through the live crop -> preprocess -> predict -> draw path, no camera needed."""
    frame = cv2.cvtColor(cv2.resize(image, (CONFIG["CAM_HEIGHT"], CONFIG["CAM_HEIGHT"]),
                                    interpolation=cv2.INTER_CUBIC), cv2.COLOR_GRAY2BGR)
    frame = cv2.flip(frame, 1)                           # pretend it is the mirrored display frame
    h, w = frame.shape[:2]
    # Two fake "landmarks" spanning the middle 60 % of the frame stand in for a detected hand.
    fake_landmarks = np.array([[w * 0.2, h * 0.2], [w * 0.8, h * 0.8]], dtype=np.float32)
    box = landmarks_to_square_box(fake_landmarks, frame.shape)
    img28 = crop_to_model_image(frame, box, mirror_input=False, clahe=None)
    smoother = PredictionSmoother()
    smoother.add(predict_probabilities(net, img28))
    letter, confidence, top = smoother.result()
    cv2.rectangle(frame, box[:2], box[2:], GREEN, 2)
    draw_inset(frame, img28)
    panel = render_panel(w, letter, confidence, top, "DEMO", 0.5, 30.0, str(DEVICE), False, False)

    fig, ax = plt.subplots(figsize=(6, 7.5))
    ax.imshow(cv2.cvtColor(np.vstack([frame, panel]), cv2.COLOR_BGR2RGB))
    ax.set_title(f"Dry run on a test image (true letter {true_letter})")
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
    plt.show()
    print(f"Prediction: {letter} ({confidence:.1%}), top-3 {[(l, round(p, 4)) for l, p in top]}, box {box}")


# Skipped if the dataset was not loaded in this session (e.g. when running the live section on its own).
if "test_images" in globals():
    dry_run_preview(live_model, test_images[0], IDX_TO_LETTER[int(test_labels[0])])
else:
    print("Dataset not loaded in this session; skipping the dry run.")

**Interpreting the output.** The dry run sends a test image through exactly the live code path (mirroring, cropping, grayscale, resizing, scaling, inference, smoothing and drawing) without a camera. The prediction should match the true letter, and the preview shows what the live window looks like. If this cell fails, the problem is in the code rather than in the camera or the lighting.

### 16.4 The live loop

For each frame: read it, mirror it for display, detect the hand, draw the landmarks, crop and preprocess, run the CNN inside `torch.inference_mode()` on the GPU, apply softmax, copy only the 24 probabilities back to the CPU, smooth them, update the word builder, draw everything and handle keys.

| Key | Action |
|---|---|
| **q** | quit |
| **m** | toggle mirroring of the model input |
| **h** | toggle CLAHE contrast enhancement |
| **space** | add a space to the text |
| **backspace** | delete the last character |
| **c** | clear the text |

The window must have keyboard focus for the keys to work. The camera is released and the window destroyed in a `finally` block, so even if something crashes mid-loop the webcam is not left locked (otherwise you would need to restart the kernel before you could open it again).

**Tips for good live predictions:** a plain, light background behind the hand; even light from the front; the hand roughly upright and filling the green box; hold each letter still. Watch the inset: it should look like the dataset samples.

In [ ]:
def run_live_demo(net: nn.Module, cam_index: int = CAM_INDEX, cfg: dict = CONFIG,
                  device: torch.device = DEVICE) -> str:
    """Run live letter recognition in an OpenCV window until q is pressed. Returns the typed text."""
    cap = cv2.VideoCapture(cam_index)
    if not cap.isOpened():
        print(f"Could not open camera {cam_index}. Check that no other app (Zoom, Teams, a browser tab, "
              "another notebook) is using it, that camera access is allowed in your OS privacy settings, "
              "and try another CAM_INDEX (0, 1, 2 ...).")
        return ""
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, cfg["CAM_WIDTH"])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cfg["CAM_HEIGHT"])

    detector = None
    words = WordBuilder(cfg["HOLD_SECONDS"])
    try:
        detector = HandDetector(cfg, video_mode=True)
        clahe_op = cv2.createCLAHE(clipLimit=cfg["CLAHE_CLIP_LIMIT"],
                                   tileGridSize=(cfg["CLAHE_TILE_GRID"], cfg["CLAHE_TILE_GRID"]))
        smoother = PredictionSmoother(cfg["SMOOTHING_WINDOW"], cfg["CONF_THRESHOLD"])
        mirror_input, use_clahe = cfg["MIRROR_INPUT"], cfg["USE_CLAHE"]
        net.eval()                                   # once, outside the loop
        fps, t_prev = 0.0, time.perf_counter()
        print(f"Live demo running with {detector.backend} on {device}. Press q in the video window to quit.")

        while True:
            ok, frame = cap.read()
            if not ok:
                print("Could not read a frame from the camera; stopping.")
                break
            display = cv2.flip(frame, 1)             # mirror for a natural, mirror-like view
            hands = detector.detect(display)

            if hands:
                points = hands[0]
                draw_landmarks(display, points)
                box = landmarks_to_square_box(points, display.shape, cfg["BOX_PADDING"])
                img28 = crop_to_model_image(display, box, mirror_input, clahe_op if use_clahe else None)
                smoother.add(predict_probabilities(net, img28, device))
                letter, confidence, top = smoother.result(cfg["TOP_K"])
                cv2.rectangle(display, box[:2], box[2:], GREEN, 2)
                draw_inset(display, img28, cfg)
            else:
                smoother.clear()                     # stale predictions must not leak into the next hand
                letter, confidence, top = None, 0.0, []

            now = time.perf_counter()
            hold = words.update(letter, now)
            fps = cfg["FPS_SMOOTHING"] * fps + (1 - cfg["FPS_SMOOTHING"]) * (1.0 / max(now - t_prev, 1e-6))
            t_prev = now

            panel = render_panel(display.shape[1], letter, confidence, top, words.text, hold, fps,
                                 str(device), mirror_input, use_clahe, cfg)
            cv2.imshow(cfg["WINDOW_NAME"], np.vstack([display, panel]))

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("m"):
                mirror_input = not mirror_input
                smoother.clear()
            elif key == ord("h"):
                use_clahe = not use_clahe
                smoother.clear()
            elif key == ord("c"):
                words.clear()
            elif key == ord(" "):
                words.space()
            elif key in (8, 127):                    # backspace (8 on Windows/Linux, 127 on macOS)
                words.backspace()
            if cv2.getWindowProperty(cfg["WINDOW_NAME"], cv2.WND_PROP_VISIBLE) < 1:
                break                                # window closed with the mouse
    finally:
        cap.release()
        if detector is not None:
            detector.close()
        cv2.destroyAllWindows()
        cv2.waitKey(1)                               # lets the window actually close on some platforms
    print(f"Camera released. Typed text: {words.text!r}")
    return words.text

In [ ]:
if CONFIG["RUN_LIVE_DEMO"]:
    typed_text = run_live_demo(live_model)
else:
    print("Live demo skipped (CONFIG['RUN_LIVE_DEMO'] is False).")

**What to expect.** On a laptop with a CUDA GPU the loop typically runs at the camera's frame rate (about 30 FPS); MediaPipe usually costs more time than the CNN. Letters with distinctive shapes (B, C, L, V, W, Y) usually work well straight away. The fist-based letters (A, E, M, N, S, T) are the hardest, as on the test set, and are also the most sensitive to lighting and framing.

If live accuracy is much worse than the test accuracy, that is the *domain gap*: your webcam, your hand, your background and your lighting differ from the dataset's. Use the inset to narrow it down: try **m** and **h**, adjust `BOX_PADDING`, improve the lighting, and if that is not enough, collect some of your own images and fine-tune (section 17).

## 17. Optional: fine-tuning on your own images

**What.** Record about 150 crops per letter of *your* hand with *your* webcam, using exactly the same detection, crop and preprocessing code as the live demo, then continue training the existing model for a few epochs on a mix of the original training data and your images.

**Why it helps.** The live demo meets images from a slightly different distribution than the training set (different camera, hand, background, lighting): a *domain gap*. Fine-tuning on a modest number of in-domain examples moves the decision boundaries towards your conditions. Three choices keep it safe:

- a **low learning rate** (1e-4 instead of 1e-3), so the model adapts rather than relearning from scratch;
- **mixing in original training images**, so it does not forget what it already knew (*catastrophic forgetting*), which we check on the original validation split;
- the **same GPU augmentation**, so a small set of your images still produces varied training batches.

Your images are split three ways: training, a validation part used for early stopping, and a **held-out part used only for the final report**. The held-out score is the most honest estimate in this notebook of how well the model will work for you live.

### 17.1 Capture

Set `CONFIG["RUN_CAPTURE"] = True` and run the next cell. For each letter the window tells you which letter to show. Press **space** to start recording (it saves every second frame until it has `CAPTURE_PER_LETTER` images, then stops), **n** for the next letter, **p** for the previous one, and **q** to quit. Move your hand slightly while recording (distance, angle, position) so the images vary. A reference chart of the letters is in `dataset/amer_sign2.png`.

Images are saved as the 28 x 28 model input (after mirroring choice and CLAHE, if on), as PNG files in `my_data/<letter>/`. Running the capture again adds to the existing images.

In [ ]:
def run_capture_session(cam_index: int = CAM_INDEX, cfg: dict = CONFIG) -> None:
    """Record 28 x 28 model-input crops of your hand into MY_DATA_DIR/<letter>/ using the live pipeline."""
    cap = cv2.VideoCapture(cam_index)
    if not cap.isOpened():
        print(f"Could not open camera {cam_index}; check CAM_INDEX and that no other app is using it.")
        return
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, cfg["CAM_WIDTH"])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cfg["CAM_HEIGHT"])
    root = Path(cfg["MY_DATA_DIR"])
    letters = list(cfg["CAPTURE_LETTERS"])
    clahe_op = cv2.createCLAHE(clipLimit=cfg["CLAHE_CLIP_LIMIT"],
                               tileGridSize=(cfg["CLAHE_TILE_GRID"], cfg["CLAHE_TILE_GRID"]))
    detector = None
    li, recording, frame_count, saved_now = 0, False, 0, 0
    try:
        detector = HandDetector(cfg, video_mode=True)
        while True:
            letter = letters[li]
            folder = root / letter
            folder.mkdir(parents=True, exist_ok=True)
            n_existing = len(list(folder.glob("*.png")))

            ok, frame = cap.read()
            if not ok:
                print("Could not read a frame; stopping.")
                break
            display = cv2.flip(frame, 1)
            hands = detector.detect(display)
            if hands:
                draw_landmarks(display, hands[0])
                box = landmarks_to_square_box(hands[0], display.shape, cfg["BOX_PADDING"])
                img28 = crop_to_model_image(display, box, cfg["MIRROR_INPUT"],
                                            clahe_op if cfg["USE_CLAHE"] else None)
                cv2.rectangle(display, box[:2], box[2:], RED if recording else GREEN, 2)
                draw_inset(display, img28, cfg)
                frame_count += 1
                if recording and frame_count % cfg["CAPTURE_EVERY_N_FRAMES"] == 0:
                    cv2.imwrite(str(folder / f"{letter}_{time.time_ns()}.png"), img28)
                    saved_now += 1
                    if saved_now >= cfg["CAPTURE_PER_LETTER"]:
                        recording = False
                        print(f"{letter}: recorded {saved_now} images ({n_existing + 1} in total).")

            panel = np.zeros((cfg["PANEL_HEIGHT"], display.shape[1], 3), dtype=np.uint8)
            cv2.putText(panel, f"Show letter: {letter}", (20, 60), FONT, 1.4, GREEN, 3, cv2.LINE_AA)
            state = f"RECORDING {saved_now}/{cfg['CAPTURE_PER_LETTER']}" if recording else "paused"
            cv2.putText(panel, f"{state} | saved for {letter}: {n_existing}", (20, 105), FONT, 0.8,
                        RED if recording else WHITE, 2, cv2.LINE_AA)
            cv2.putText(panel, "space start/stop  n next  p previous  q quit", (20, 150), FONT, 0.65, GREY, 1, cv2.LINE_AA)
            cv2.imshow(cfg["WINDOW_NAME"], np.vstack([display, panel]))

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord(" "):
                recording, saved_now = not recording, 0
            elif key in (ord("n"), ord("p")):
                li = (li + (1 if key == ord("n") else -1)) % len(letters)
                recording, saved_now = False, 0
    finally:
        cap.release()
        if detector is not None:
            detector.close()
        cv2.destroyAllWindows()
        cv2.waitKey(1)
    print("Capture finished; camera released.")


if CONFIG["RUN_CAPTURE"]:
    run_capture_session()
else:
    print("Capture skipped (set CONFIG['RUN_CAPTURE'] = True to record your own images).")

### 17.2 Fine-tune

The next cell loads your images (it skips fine-tuning with a message if there are fewer than `FT_MIN_IMAGES`), splits them, measures the current model on your held-out images, fine-tunes a fresh copy of the saved model with `fit()` from section 10, and reports:

- accuracy on your held-out images **before and after** fine-tuning (the improvement you should feel live);
- accuracy on the original validation split before and after (it should barely change; a big drop would mean forgetting).

The mixed training set contains each of your training images `FT_MY_REPEAT` times plus `FT_ORIG_PER_MY` randomly chosen original training images for each of those. All of it lives on the GPU and is augmented per batch as before.

In [ ]:
def load_my_images(root: Path, letter_to_idx: dict[str, int]) -> tuple[np.ndarray, np.ndarray]:
    """Read root/<letter>/*.png as 28 x 28 uint8 grayscale images with class indices."""
    size = CONFIG["IMG_SIZE"]
    images, labels = [], []
    for letter, idx in letter_to_idx.items():
        for path in sorted((root / letter).glob("*.png")):
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            if img.shape != (size, size):
                img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
            images.append(img)
            labels.append(idx)
    if not images:
        return np.zeros((0, size, size), np.uint8), np.zeros(0, np.int64)
    return np.stack(images), np.array(labels, dtype=np.int64)


def accuracy_on(net: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    return (predict_logits(net, X).argmax(1) == y).float().mean().item()


finetuned_model = None
my_images, my_labels = load_my_images(Path(CONFIG["MY_DATA_DIR"]), LETTER_TO_IDX)
print(f"Found {len(my_images)} of your images for {len(np.unique(my_labels))} letters in {CONFIG['MY_DATA_DIR']}/")

if not CONFIG["RUN_FINETUNE"]:
    print("Fine-tuning skipped (CONFIG['RUN_FINETUNE'] is False).")
elif len(my_images) < CONFIG["FT_MIN_IMAGES"]:
    print(f"Fine-tuning skipped: need at least {CONFIG['FT_MIN_IMAGES']} images. Record some with the capture cell first.")
elif np.bincount(my_labels).max() < 3 or (np.bincount(my_labels)[np.unique(my_labels)] < 3).any():
    print("Fine-tuning skipped: every captured letter needs at least 3 images for a stratified split.")
else:
    seed_everything(CONFIG["SEED"])
    idx_all = np.arange(len(my_labels))
    idx_rest, idx_hold = train_test_split(idx_all, test_size=CONFIG["FT_HOLDOUT_FRAC"],
                                          stratify=my_labels, random_state=CONFIG["SEED"])
    idx_ft_train, idx_ft_val = train_test_split(idx_rest, test_size=CONFIG["FT_VAL_FRAC"],
                                                stratify=my_labels[idx_rest], random_state=CONFIG["SEED"])

    def to_device(idx: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
        return scale_pixels(my_images[idx]).unsqueeze(1), torch.from_numpy(my_labels[idx]).to(DEVICE)

    X_my_train, y_my_train = to_device(idx_ft_train)
    X_my_val, y_my_val = to_device(idx_ft_val)
    X_my_hold, y_my_hold = to_device(idx_hold)
    print(f"Your images: {len(idx_ft_train)} train, {len(idx_ft_val)} validation, {len(idx_hold)} held out")

    # Mix: your images repeated FT_MY_REPEAT times + FT_ORIG_PER_MY original images per repeated image.
    X_mine_rep = X_my_train.repeat(CONFIG["FT_MY_REPEAT"], 1, 1, 1)
    y_mine_rep = y_my_train.repeat(CONFIG["FT_MY_REPEAT"])
    n_orig = min(len(X_train), CONFIG["FT_ORIG_PER_MY"] * len(X_mine_rep))
    orig_idx = torch.randperm(len(X_train), device=DEVICE)[:n_orig]
    X_mix = torch.cat([X_mine_rep, X_train[orig_idx]])
    y_mix = torch.cat([y_mine_rep, y_train[orig_idx]])
    print(f"Fine-tuning set: {len(X_mix)} images ({len(X_mine_rep)} yours incl. repeats, {n_orig} original)")

    base = load_model()
    hold_before, val_before = accuracy_on(base, X_my_hold, y_my_hold), accuracy_on(base, X_val, y_val)

    finetuned_model = load_model()
    ft_history, ft_best = fit(finetuned_model, X_mix, y_mix, X_my_val, y_my_val, lr=CONFIG["FT_LR"],
                              max_epochs=CONFIG["FT_EPOCHS"], es_patience=CONFIG["FT_PATIENCE"], augment=True)
    finetuned_model.eval()
    hold_after, val_after = accuracy_on(finetuned_model, X_my_hold, y_my_hold), accuracy_on(finetuned_model, X_val, y_val)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print(f"\nYour held-out images : {hold_before:.2%} -> {hold_after:.2%}")
    print(f"Original validation  : {val_before:.2%} -> {val_after:.2%}  (should stay about the same)")
    ft_path = Path(CONFIG["OUTPUT_DIR"]) / CONFIG["FINETUNED_MODEL_FILE"]
    torch.save(finetuned_model.state_dict(), ft_path)
    print(f"Saved the fine-tuned weights to {ft_path}")

    ft_df = pd.DataFrame(ft_history).set_index("epoch")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ft_df.index, ft_df["train_loss"], label="train (mixed, augmented)")
    ax.plot(ft_df.index, ft_df["val_loss"], label="validation (your images)")
    ax.axvline(ft_best, color="grey", linestyle="--", label=f"restored epoch {ft_best}")
    ax.set_title("Fine-tuning loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy")
    ax.legend()
    plt.show()

In [ ]:
# Try the fine-tuned model live (only if fine-tuning ran above).
if finetuned_model is None:
    print("No fine-tuned model in this session; skipping the second live demo.")
elif not CONFIG["RUN_LIVE_DEMO"]:
    print("Live demo skipped (CONFIG['RUN_LIVE_DEMO'] is False).")
else:
    typed_text = run_live_demo(finetuned_model)

**Interpreting the output.** In a test of this pipeline with 720 captured crops (30 per letter), held-out accuracy rose from about 96 % to 100 % after fine-tuning and the original validation accuracy stayed at 100 %. With your own varied captures, accuracy on your held-out images should rise noticeably after fine-tuning while accuracy on the original validation split stays roughly where it was, meaning the model has adapted to your conditions without forgetting the dataset. Be aware that your held-out images were recorded in the same session as your training images, so they share lighting and background; recording a second session on a different day and evaluating on that would be a stricter test.

## 18. Conclusion

The cell below collects the headline numbers from this run.

In [ ]:
summary = {
    "device": f"{DEVICE} ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else str(DEVICE),
    "epochs trained / restored epoch": f"{len(history)} / {best_epoch}",
    "total training time (s)": f"{total_train_time:.0f}",
    "fit verdict": fit_report["verdict"],
    "train-val gap (points)": f"{fit_report['gap'] * 100:+.2f}",
    "baseline test accuracy": f"{baseline_test_acc:.2%}",
    "CNN test accuracy": f"{test_acc:.2%}",
    "CNN test macro F1": f"{test_macro_f1:.4f}",
    "worst robustness condition": f"{robust_df.loc[robust_df['accuracy'].idxmin(), 'condition']} "
                                  f"({robust_df['accuracy'].min():.2%})",
    "most confused pair": f"{confused.loc[0, 'true']} -> {confused.loc[0, 'predicted']} "
                          f"({confused.loc[0, 'errors']} errors, mean prob {confused.loc[0, 'mean_prob_given']:.3f})",
}
pd.Series(summary, name="value").to_frame()

### Summary

A compact three-block CNN (about 0.8 M parameters), trained in under a minute on a laptop GPU with per-image GPU augmentation, mixed precision and a hand-written early-stopping loop, exceeds the project targets on the official Sign Language MNIST test set. In the reference run it reached 100.00 % accuracy and macro F1 1.000, against 62.8 % for the logistic-regression baseline. Training and validation curves agree (gap 0 points, verdict "good fit"), and Grad-CAM shows the network attending to the hand rather than the background. The robustness check shows the model handles brightness changes and rotations well but is sensitive to strong blur and noise; the resize from webcam resolution to 28 x 28 largely averages the noise away, but not motion blur. The same preprocessing function feeds the model during training and live, and the live demo adds hand detection, square cropping, temporal smoothing and a confidence threshold to turn per-frame predictions into a usable letter display.

### Limitations

- **J and Z are missing.** They are defined by motion, which a single-frame classifier cannot see.
- **Lighting.** The robustness check shows accuracy falling under strong brightness changes. Dim or one-sided light makes the fist letters especially unreliable. CLAHE helps, but is not a cure.
- **Background.** The dataset has plain backgrounds close to the hand. A busy background inside the crop is new to the model.
- **Handedness and mirroring.** Most dataset signs appear right-handed from the camera's viewpoint. Left-handed signers need `MIRROR_INPUT` (or a model trained with `AUG_HFLIP`, at the cost of confusing orientation-dependent letters).
- **The dataset-to-webcam gap.** The dataset was built from a small number of original photos, heavily augmented, and the near-duplicates make the validation score optimistic. Even the test score is measured on images like the training images, not on webcam crops, so real-world accuracy is lower. The fine-tuning held-out score is the best estimate of live performance.
- **Low resolution.** At 28 x 28 the thumb position that separates M, N, S and T covers only a few pixels.
- **A ceiling effect.** Because the test score is already 100 %, the test set can no longer tell better models from worse ones. Improvements now have to be measured on harder data, such as degraded images, your own webcam captures, or other people's hands.

### Next steps

- Fine-tune on images from several people, sessions and rooms, and evaluate on a person not in the training data.
- Train at a higher resolution on a richer dataset, or use a pretrained backbone (transfer learning).
- Classify from the **21 MediaPipe landmarks** instead of (or as well as) pixels: landmark coordinates are largely invariant to lighting and background, which addresses the main live failure modes.
- Add a sequence model over landmark trajectories (for example a small GRU or temporal CNN) to recognise **J and Z** and, later, whole signs.
- Add a language model or dictionary to correct the typed words.